# 📈 Stock Price Prediction — ML Pipeline v3.0 (Production-Ready)
**All bugs fixed · Real-time prices · Indian stocks · Live Gradio feed · Zero paid APIs**

> Trained on 2015–present data.
> The model's `predict_tomorrow()` uses yfinance which lags by 1 trading day (last close).  
> The Gradio UI polls yfinance every 30 s for a live price ticker.


In [1]:
!pip install --upgrade yfinance -q
!pip install ta xgboost plotly scikit-learn tensorflow seaborn \
             transformers torch optuna shap lightgbm feedparser fredapi \
             requests beautifulsoup4 groq gradio scipy langgraph langchain-groq -q

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, yfinance as yf, json, os, joblib, time, threading
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import feedparser, requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from scipy import stats
from ta.trend import MACD, EMAIndicator, SMAIndicator
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
    r2_score, confusion_matrix, ConfusionMatrixDisplay)
import xgboost as xgb, lightgbm as lgb
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
print('✅ All libraries loaded')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.5 MB/s eta 0:00:00
✅ All libraries loaded


## ⚙️ Configuration

In [2]:
CONFIG = {
    'ticker'          : 'AAPL',
    'start_date'      : '2015-01-01',
    'end_date'        : datetime.today().strftime('%Y-%m-%d'),
    'test_size'       : 0.20,
    'look_back'       : 60,
    'forecast_days'   : 1,
    'random_state'    : 42,
    'n_folds'         : 5,
    'multistep_days'  : [1, 3, 5, 10],
    'initial_capital' : 10_000.0,
    'transaction_cost': 0.001,
    # Indian NSE tickers use .NS suffix — e.g. 'RELIANCE.NS', 'TCS.NS', 'INFY.NS'
    # BSE tickers use .BO suffix — e.g. 'RELIANCE.BO'
    'indian_tickers'  : ['RELIANCE.NS','TCS.NS','INFY.NS','HDFCBANK.NS','WIPRO.NS'],
}
np.random.seed(CONFIG['random_state'])
tf.random.set_seed(CONFIG['random_state'])
print(f"✅ Config loaded | {CONFIG['ticker']} | {CONFIG['start_date']} → {CONFIG['end_date']}")


✅ Config loaded | AAPL | 2015-01-01 → 2026-06-19


## 🔑 API Keys (All Free)
| Key | Source | Status |
|---|---|---|
| `GROQ_API_KEY` | console.groq.com | Optional — LLM report |
| `FRED_API_KEY` | fred.stlouisfed.org | Optional — macro features |
| `ALPHAVANTAGE_API_KEY` | alphavantage.co | Optional — news sentiment |
| `FINNHUB_API_KEY` | finnhub.io | Optional — company news |


In [3]:
try:
    from google.colab import userdata
    KEYS = {
        'groq'        : userdata.get('GROQ_API_KEY'),
        'fred'        : userdata.get('FRED_API_KEY'),
        'alphavantage': userdata.get('ALPHAVANTAGE_API_KEY'),
        'finnhub'     : userdata.get('FINNHUB_API_KEY'),
    }
    print('✅ Keys loaded from Colab Secrets')
except Exception:
    KEYS = {'groq': '', 'fred': '', 'alphavantage': '', 'finnhub': ''}
    print('⚠️  No Colab Secrets found — add keys manually to KEYS dict')
for k, v in KEYS.items():
    print(f'  {k:18s}: {"✅ active" if v else "❌ not set"}')


✅ Keys loaded from Colab Secrets
  groq              : ✅ active
  fred              : ✅ active
  alphavantage      : ✅ active
  finnhub           : ✅ active


# **api test**

In [4]:
import requests
from datetime import datetime, timedelta

print("=" * 55)
print("  API DIAGNOSTICS")
print("=" * 55)

# ── 1. ALPHA VANTAGE ──────────────────────────────────────
print("\n📊 ALPHA VANTAGE")
key_av = KEYS.get('alphavantage', '')
print(f"  Key loaded: {'✅ ' + key_av[:6] + '...' if key_av else '❌ not set — skipped'}")
if key_av:
    try:
        r = requests.get(
            f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT"
            f"&tickers=AAPL&limit=5&apikey={key_av}",
            timeout=15
        )
        d = r.json()
        if 'Information' in d:
            print(f"  ⚠️  Rate limit / plan wall: {d['Information'][:120]}")
        elif 'Note' in d:
            print(f"  ⚠️  API note: {d['Note'][:120]}")
        elif 'Error Message' in d:
            print(f"  ❌ Error: {d['Error Message']}")
        elif 'feed' in d:
            print(f"  ✅ Working — got {len(d['feed'])} articles")
        else:
            print(f"  ❓ Unknown response keys: {list(d.keys())}")
    except Exception as e:
        print(f"  ❌ Request failed: {e}")

# ── 2. FINNHUB ────────────────────────────────────────────
print("\n📡 FINNHUB")
key_fh = KEYS.get('finnhub', '')
print(f"  Key loaded: {'✅ ' + key_fh[:6] + '...' if key_fh else '❌ not set — skipped'}")
if key_fh:
    # Test 1: live quote
    try:
        r = requests.get(
            f"https://finnhub.io/api/v1/quote?symbol=AAPL&token={key_fh}",
            timeout=10
        )
        d = r.json()
        if d.get('c', 0) > 0:
            print(f"  ✅ Quote API working — AAPL current price: ${d['c']:.2f}")
        elif 'error' in d:
            print(f"  ❌ Quote error: {d['error']}")
        else:
            print(f"  ⚠️  Quote returned zero/empty: {d}")
    except Exception as e:
        print(f"  ❌ Quote request failed: {e}")

    # Test 2: company news
    try:
        end_dt   = datetime.today().strftime('%Y-%m-%d')
        start_dt = (datetime.today() - timedelta(days=7)).strftime('%Y-%m-%d')
        r = requests.get(
            f"https://finnhub.io/api/v1/company-news?symbol=AAPL"
            f"&from={start_dt}&to={end_dt}&token={key_fh}",
            timeout=10
        )
        d = r.json()
        if isinstance(d, list):
            print(f"  ✅ News API working — got {len(d)} articles (last 7 days)")
        elif isinstance(d, dict) and 'error' in d:
            print(f"  ❌ News error: {d['error']}")
        else:
            print(f"  ⚠️  News unexpected response: {d}")
    except Exception as e:
        print(f"  ❌ News request failed: {e}")

# ── 3. FRED ───────────────────────────────────────────────
print("\n🏦 FRED (Federal Reserve)")
key_fr = KEYS.get('fred', '')
print(f"  Key loaded: {'✅ ' + key_fr[:6] + '...' if key_fr else '❌ not set — skipped'}")
if key_fr:
    try:
        r = requests.get(
            f"https://api.stlouisfed.org/fred/series/observations"
            f"?series_id=DFF&limit=3&sort_order=desc"
            f"&api_key={key_fr}&file_type=json",
            timeout=10
        )
        d = r.json()
        if 'observations' in d:
            latest = d['observations'][0]
            print(f"  ✅ Working — Fed Funds Rate: {latest['value']}% (as of {latest['date']})")
        elif 'error_message' in d:
            print(f"  ❌ Error: {d['error_message']}")
        else:
            print(f"  ⚠️  Unexpected response keys: {list(d.keys())}")
    except Exception as e:
        print(f"  ❌ Request failed: {e}")

print("\n" + "=" * 55)
print("  Keys not set = feature silently skipped (not a crash)")
print("  All 3 are optional — yfinance covers the core data")
print("=" * 55)

  API DIAGNOSTICS

📊 ALPHA VANTAGE
  Key loaded: ✅ 5E5B1K...
  ✅ Working — got 50 articles

📡 FINNHUB
  Key loaded: ✅ d8qbvg...
  ✅ Quote API working — AAPL current price: $298.01
  ✅ News API working — got 248 articles (last 7 days)

🏦 FRED (Federal Reserve)
  Key loaded: ✅ 65145e...
  ✅ Working — Fed Funds Rate: 3.63% (as of 2026-06-17)

  Keys not set = feature silently skipped (not a crash)
  All 3 are optional — yfinance covers the core data


## 📡 Live Price Feed Utilities
These functions power the **real-time ticker** in Gradio.  
`get_live_price()` hits yfinance's fast_info endpoint — no API key, no rate limit for normal use.  
For Indian stocks use `.NS` (NSE) or `.BO` (BSE) suffix.


In [5]:
# ─────────────────────────────────────────────────────────
# LIVE PRICE FEED — Agentic Multi-Source Architecture
# Sources tried in parallel; best result wins automatically.
#
# US stocks  → Finnhub REST (free key, finnhub.io)
# Indian .NS → NSE India public API (no key, real-time)
# Fallback   → yfinance 5-day daily (no key, always works)
# ─────────────────────────────────────────────────────────

import concurrent.futures

# ── Source 1: NSE India official API (Indian stocks only) ─────────────────────
def _fetch_nse(symbol_clean: str) -> dict:
    """
    Hits NSE India's own quote endpoint — same feed the NSE website uses.
    Real-time during market hours 9:15–15:30 IST, no API key needed.
    symbol_clean: base symbol without suffix e.g. 'INFY' not 'INFY.NS'
    """
    headers = {
        'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Accept'         : '*/*',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer'        : 'https://www.nseindia.com',
    }
    session = requests.Session()
    # NSE blocks cold API calls — must hit homepage first to get cookies
    session.get('https://www.nseindia.com', headers=headers, timeout=10)
    r = session.get(
        f'https://www.nseindia.com/api/quote-equity?symbol={symbol_clean.upper()}',
        headers=headers, timeout=10
    )
    d = r.json()
    pi     = d['priceInfo']
    price  = float(pi['lastPrice'])
    prev   = float(pi['previousClose'])
    change = float(pi['change'])
    chgpct = float(pi['pChange'])
    try:
        vol = int(d['marketDeptOrderBook']['tradeInfo']['totalTradedVolume'])
    except Exception:
        vol = 0
    try:
        mstate = 'REGULAR' if d['marketStatus']['marketStatus'] == 'Open' else 'CLOSED'
    except Exception:
        mstate = 'CLOSED'
    return {
        'ticker'      : symbol_clean,
        'price'       : round(price, 2),
        'prev_close'  : round(prev, 2),
        'change'      : round(change, 2),
        'change_pct'  : round(chgpct, 3),
        'volume'      : vol,
        'currency'    : 'INR',
        'market_state': mstate,
        'arrow'       : '▲' if change >= 0 else '▼',
        'color_hint'  : 'green' if change >= 0 else 'red',
        'timestamp'   : datetime.now().strftime('%H:%M:%S'),
        'source'      : 'NSE_API',
        'error'       : None,
    }

# ── Source 2: Finnhub REST (US stocks, free key) ──────────────────────────────
def _fetch_finnhub(ticker: str) -> dict:
    """
    Finnhub /quote endpoint — real-time US stock prices.
    Free tier: 60 calls/minute, no daily limit.
    """
    key = KEYS.get('finnhub', '')
    if not key:
        raise ValueError('No FINNHUB_API_KEY in secrets')
    r = requests.get(
        f'https://finnhub.io/api/v1/quote?symbol={ticker}&token={key}',
        timeout=8
    ).json()
    price = r.get('c', 0)
    prev  = r.get('pc', 0)
    if not price or price == 0:
        raise ValueError(f'Finnhub returned zero price for {ticker}')
    change  = price - prev
    chgpct  = (change / prev * 100) if prev else 0
    return {
        'ticker'      : ticker,
        'price'       : round(price, 2),
        'prev_close'  : round(prev, 2),
        'change'      : round(change, 2),
        'change_pct'  : round(chgpct, 3),
        'volume'      : int(r.get('v', 0)),
        'currency'    : 'USD',
        'market_state': 'REGULAR' if r.get('d') is not None else 'CLOSED',
        'arrow'       : '▲' if change >= 0 else '▼',
        'color_hint'  : 'green' if change >= 0 else 'red',
        'timestamp'   : datetime.now().strftime('%H:%M:%S'),
        'source'      : 'Finnhub',
        'error'       : None,
    }

# ── Source 3: yfinance 5-min intraday (universal fallback) ────────────────────
def _fetch_yfinance_intraday(ticker: str) -> dict:
    """
    5-minute candles over last 2 days.
    Works for US + Indian but may lag or fail — used as fallback only.
    """
    is_indian = ticker.endswith('.NS') or ticker.endswith('.BO')
    currency  = 'INR' if is_indian else 'USD'
    df = yf.download(ticker, period='2d', interval='5m',
                     auto_adjust=True, progress=False, timeout=10)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.dropna(subset=['Close'])
    if df.empty:
        raise ValueError('yfinance intraday empty')
    price  = float(df['Close'].iloc[-1])
    volume = int(df['Volume'].iloc[-1])
    # Separate today vs yesterday bars for accurate prev_close
    today_str  = datetime.utcnow().strftime('%Y-%m-%d')
    today_bars = df[df.index.strftime('%Y-%m-%d') == today_str]
    yest_bars  = df[df.index.strftime('%Y-%m-%d') != today_str]
    if not yest_bars.empty and not today_bars.empty:
        prev_close = float(yest_bars['Close'].iloc[-1])
    else:
        prev_close = float(df['Close'].iloc[-2]) if len(df) >= 2 else price
    # Infer market state from data freshness
    age_min = (datetime.utcnow() - df.index[-1].to_pydatetime().replace(tzinfo=None)).total_seconds() / 60
    mstate  = 'REGULAR (~5m delay)' if age_min < 20 else 'CLOSED/POST'
    change  = price - prev_close
    chgpct  = (change / prev_close * 100) if prev_close else 0
    return {
        'ticker'      : ticker,
        'price'       : round(price, 2),
        'prev_close'  : round(prev_close, 2),
        'change'      : round(change, 2),
        'change_pct'  : round(chgpct, 3),
        'volume'      : volume,
        'currency'    : currency,
        'market_state': mstate,
        'arrow'       : '▲' if change >= 0 else '▼',
        'color_hint'  : 'green' if change >= 0 else 'red',
        'timestamp'   : datetime.now().strftime('%H:%M:%S'),
        'source'      : 'yfinance_intraday',
        'error'       : None,
    }

# ── Source 4: yfinance daily (last-resort fallback) ───────────────────────────
def _fetch_yfinance_daily(ticker: str) -> dict:
    """Last resort — always works but shows yesterday's close."""
    is_indian = ticker.endswith('.NS') or ticker.endswith('.BO')
    currency  = 'INR' if is_indian else 'USD'
    df = yf.download(ticker, period='5d', interval='1d',
                     auto_adjust=True, progress=False, timeout=10)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    closes    = df['Close'].dropna()
    price     = float(closes.iloc[-1])
    prev      = float(closes.iloc[-2]) if len(closes) >= 2 else price
    change    = price - prev
    chgpct    = (change / prev * 100) if prev else 0
    return {
        'ticker'      : ticker,
        'price'       : round(price, 2),
        'prev_close'  : round(prev, 2),
        'change'      : round(change, 2),
        'change_pct'  : round(chgpct, 3),
        'volume'      : int(df['Volume'].dropna().iloc[-1]),
        'currency'    : currency,
        'market_state': 'CLOSED (daily data — prev session close)',
        'arrow'       : '▲' if change >= 0 else '▼',
        'color_hint'  : 'green' if change >= 0 else 'red',
        'timestamp'   : datetime.now().strftime('%H:%M:%S'),
        'source'      : 'yfinance_daily',
        'error'       : None,
    }

# ── MASTER AGENT: fires sources in parallel, picks best result ─────────────────
def get_live_price(ticker: str) -> dict:
    """
    Agentic price fetcher — runs all relevant sources in parallel threads,
    returns the first successful non-zero result in priority order.

    Priority:
      Indian tickers (.NS/.BO) → NSE_API first, yfinance_intraday second, daily last
      US tickers               → Finnhub first, yfinance_intraday second, daily last

    This is faster than sequential fallback because all sources start at the same time.
    Whichever finishes first with a valid price wins.
    """
    is_indian     = ticker.endswith('.NS') or ticker.endswith('.BO')
    symbol_clean  = ticker.replace('.NS','').replace('.BO','')

    if is_indian:
        sources = [
            ('NSE_API',           lambda: _fetch_nse(symbol_clean)),
            ('yfinance_intraday', lambda: _fetch_yfinance_intraday(ticker)),
            ('yfinance_daily',    lambda: _fetch_yfinance_daily(ticker)),
        ]
        priority = ['NSE_API', 'yfinance_intraday', 'yfinance_daily']
    else:
        sources = [
            ('Finnhub',           lambda: _fetch_finnhub(ticker)),
            ('yfinance_intraday', lambda: _fetch_yfinance_intraday(ticker)),
            ('yfinance_daily',    lambda: _fetch_yfinance_daily(ticker)),
        ]
        priority = ['Finnhub', 'yfinance_intraday', 'yfinance_daily']

    results = {}
    errors  = {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        future_map = {executor.submit(fn): name for name, fn in sources}
        try:
            for future in concurrent.futures.as_completed(future_map, timeout=15):
                name = future_map[future]
                try:
                    res = future.result()
                    if res and res.get('price', 0) > 0:
                        results[name] = res
                except Exception as e:
                    errors[name] = str(e)
        except concurrent.futures.TimeoutError:
            pass  # use whatever results arrived before timeout

    # Return best result in priority order
    for source in priority:
        if source in results:
            r = results[source]
            print(f"  📡 {ticker}: source={source} price={r.get('currency','')} {r['price']}")
            return r

    # Everything failed
    err_summary = ' | '.join(f'{k}: {v}' for k, v in errors.items())
    return {
        'ticker'    : ticker,
        'price'     : 0,
        'currency'  : 'INR' if is_indian else 'USD',
        'error'     : f'All sources failed — {err_summary}',
        'timestamp' : datetime.now().strftime('%H:%M:%S'),
    }

# ── format_price_card — unchanged ─────────────────────────────────────────────
def format_price_card(info: dict) -> str:
    """Formats live price dict into a human-readable string for Gradio."""
    if info.get('error'):
        return f"❌ {info['ticker']}: {info['error']}"
    sym  = '₹' if info.get('currency') == 'INR' else '$'
    sign = '+' if info.get('change', 0) >= 0 else ''
    src  = info.get('source', '')
    return (
        f"{info.get('arrow','?')} {info['ticker']}  {sym}{info['price']:,.2f}  "
        f"{sign}{info.get('change',0):.2f} ({sign}{info.get('change_pct',0):.2f}%)  "
        f"| Vol: {info.get('volume',0):,}  "
        f"| {info.get('market_state','?')}  "
        f"| src: {src}  "
        f"| {info.get('timestamp','')}"
    )

# ── Smoke tests ───────────────────────────────────────────────────────────────
print("Running smoke tests (parallel fetch)...\n")
_test_us  = get_live_price('AAPL')
_test_in  = get_live_price('INFY.NS')
_test_in2 = get_live_price('RELIANCE.NS')
print(f"\n{format_price_card(_test_us)}")
print(f"{format_price_card(_test_in)}")
print(f"{format_price_card(_test_in2)}")

Running smoke tests (parallel fetch)...

  📡 AAPL: source=Finnhub price=USD 298.01
  📡 INFY.NS: source=yfinance_intraday price=INR 1054.6
  📡 RELIANCE.NS: source=yfinance_intraday price=INR 1311.6

▲ AAPL  $298.01  +2.06 (+0.70%)  | Vol: 0  | REGULAR  | src: Finnhub  | 11:27:12
▼ INFY.NS  ₹1,054.60  -73.40 (-6.51%)  | Vol: 1,507,895  | REGULAR (~5m delay)  | src: yfinance_intraday  | 11:27:13
▼ RELIANCE.NS  ₹1,311.60  -19.40 (-1.46%)  | Vol: 1,697,069  | REGULAR (~5m delay)  | src: yfinance_intraday  | 11:27:13


## 📥 Step 1 — Data Download

In [6]:
def download_stock_data(ticker, start, end):
    print(f'📡 Downloading {ticker}...')
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if df.empty: raise ValueError(f'No data for {ticker}')
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    df.columns = [c.strip() for c in df.columns]
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True); df.dropna(how='all', inplace=True)
    print(f'  ✅ {len(df)} days | Close: {df["Close"].min():.2f} – {df["Close"].max():.2f}')
    return df

def add_market_context(df, tickers=('SPY','QQQ','XLK','^VIX')):
    """
    Adds US market ETFs and VIX as additional features.
    For Indian stocks, also adds NIFTY50 (^NSEI) and SENSEX (^BSESN).
    """
    enriched = df.copy()
    is_indian = any(s in df.columns.get_level_values(0) if isinstance(df.columns, pd.MultiIndex)
                    else s in str(df.index.name or '') for s in ['.NS','.BO'])
    extra = ('^NSEI', '^BSESN') if is_indian else ()
    all_tickers = list(tickers) + list(extra)
    for t in all_tickers:
        try:
            col = t.replace('^','')
            mkt = yf.download(t, start=CONFIG['start_date'], end=CONFIG['end_date'],
                              auto_adjust=True, progress=False)['Close']
            if isinstance(mkt, pd.DataFrame): mkt = mkt.squeeze()
            mkt.index = pd.to_datetime(mkt.index)
            enriched[f'{col}_Close']  = mkt.reindex(enriched.index, method='ffill')
            enriched[f'{col}_Return'] = enriched[f'{col}_Close'].pct_change()
            enriched[f'{col}_Lag1']   = enriched[f'{col}_Return'].shift(1)
            print(f'  ✅ {t}: {enriched[f"{col}_Return"].notna().sum()} rows')
        except Exception as e:
            print(f'  ⚠️ {t}: {e}')
    return enriched

df_raw          = download_stock_data(CONFIG['ticker'], CONFIG['start_date'], CONFIG['end_date'])
df_raw_enriched = add_market_context(df_raw)
print(f'Shape: {df_raw.shape} → {df_raw_enriched.shape}')


📡 Downloading AAPL...
  ✅ 2882 days | Close: 20.57 – 315.20
  ✅ SPY: 2881 rows
  ✅ QQQ: 2881 rows
  ✅ XLK: 2881 rows
  ✅ ^VIX: 2881 rows
Shape: (2882, 5) → (2882, 17)


## 🤖 Step 2 — Agent 1: Live Data Scraper

In [7]:
from transformers import pipeline as hf_pipeline

print('🧠 Loading FinBERT...')
try:
    finbert = hf_pipeline('text-classification', model='ProsusAI/finbert',
                           tokenizer='ProsusAI/finbert', device=-1,
                           truncation=True, max_length=512)
    print('✅ FinBERT ready')
except Exception as e:
    finbert = None; print(f'⚠️ FinBERT unavailable: {e}')

def fetch_google_news_rss(ticker, max_articles=25):
    try:
        # Works for Indian stocks too — just pass 'RELIANCE NSE' or 'TCS India'
        url = f'https://news.google.com/rss/search?q={ticker}+stock&hl=en-IN&gl=IN&ceid=IN:en'
        feed = feedparser.parse(url)
        return [{'source':'GoogleNews','text':e.title,'score':1,'pre_scored':False}
                for e in feed.entries[:max_articles] if len(e.title) > 20]
    except Exception as e: print(f'  Google RSS: {e}'); return []

def scrape_yahoo_news(ticker, max_articles=25):
    try:
        url = f'https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker}&region=US&lang=en-US'
        feed = feedparser.parse(url)
        return [{'source':'YahooRSS','text':e.title,'score':1,'pre_scored':False}
                for e in feed.entries[:max_articles] if len(e.get('title','')) > 20]
    except Exception as e: print(f'  Yahoo RSS: {e}'); return []

def fetch_finnhub_news(ticker, max_articles=25):
    if not KEYS.get('finnhub'): return []
    try:
        end_dt   = datetime.today().strftime('%Y-%m-%d')
        start_dt = (datetime.today()-timedelta(days=7)).strftime('%Y-%m-%d')
        r = requests.get(
            f'https://finnhub.io/api/v1/company-news?symbol={ticker}'
            f'&from={start_dt}&to={end_dt}&token={KEYS["finnhub"]}', timeout=10)
        return [{'source':'Finnhub','text':a['headline'],'score':1,'pre_scored':False}
                for a in r.json()[:max_articles] if len(a.get('headline','')) > 20]
    except Exception as e: print(f'  Finnhub: {e}'); return []

def fetch_alphavantage_news(ticker, max_articles=25):
    if not KEYS.get('alphavantage'): return []
    try:
        r = requests.get(
            f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT'
            f'&tickers={ticker}&limit={max_articles}&apikey={KEYS["alphavantage"]}', timeout=15)
        feed = r.json().get('feed', [])
        return [{'source':'AlphaVantage','text':a['title'],
                 'score':float(next((s for s in a.get('ticker_sentiment',[])
                                     if s['ticker']==ticker), {}).get('ticker_sentiment_score',0)),
                 'pre_scored':True}
                for a in feed]
    except Exception as e: print(f'  AlphaVantage: {e}'); return []

def fetch_macro_features(start_date, end_date):
    if not KEYS.get('fred'): return pd.DataFrame()
    try:
        from fredapi import Fred
        f = Fred(api_key=KEYS['fred'])
        data = [f.get_series(k, observation_start=start_date, observation_end=end_date).rename(v)
                for k,v in {'DFF':'Fed_Funds_Rate','T10Y2Y':'Yield_Curve',
                             'VIXCLS':'VIX_FRED','CPIAUCSL':'CPI'}.items()]
        macro = pd.concat(data, axis=1)
        macro.index = pd.to_datetime(macro.index)
        return macro.resample('B').last().ffill()
    except Exception as e: print(f'  FRED: {e}'); return pd.DataFrame()

def scrape_analyst_rating(ticker):
    score_map = {'strong buy':2.0,'buy':1.5,'overweight':1.0,'hold':0.0,
                 'neutral':0.0,'underweight':-1.0,'sell':-1.5,'strong sell':-2.0}
    try:
        r = requests.get(f'https://finviz.com/quote.ashx?t={ticker}',
                          headers={'User-Agent':'Mozilla/5.0'}, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        for td in soup.find_all('td'):
            if 'Recom' in td.get_text():
                v = td.find_next_sibling('td')
                if v: return score_map.get(v.get_text(strip=True).lower(), 0.0)
    except Exception as e: print(f'  Finviz: {e}')
    return 0.0

def score_sentiment(texts):
    if not texts: return 0.0
    pre = [t for t in texts if t.get('pre_scored')]
    if pre: return float(np.mean([t['score'] for t in pre]))
    if finbert is None: return 0.0
    label_map = {'positive':1.0,'neutral':0.0,'negative':-1.0}
    scores = []
    for item in texts:
        try:
            res = finbert(item['text'][:512])[0]
            scores.append(label_map.get(res['label'],0.0)*res['score']*(1+np.log1p(item.get('score',1))))
        except: pass
    return float(np.mean(scores)) if scores else 0.0

def run_data_agent(df, ticker):
    print(f'\n🤖 AGENT 1: {ticker}\n' + '='*55)
    enriched = df.copy()
    yahoo  = scrape_yahoo_news(ticker)
    alt    = (fetch_finnhub_news(ticker) or fetch_alphavantage_news(ticker)
               or fetch_google_news_rss(ticker))
    ns = score_sentiment(yahoo); als = score_sentiment(alt)
    non_zero = [s for s in [ns, als] if s != 0.0]
    combined = float(np.mean(non_zero)) if non_zero else 0.0
    analyst  = scrape_analyst_rating(ticker)

    # Exponential decay over last 30 rows
    n    = min(30, len(enriched))
    decay = np.exp(-np.arange(n)[::-1] / 10.0); decay /= decay.max()
    enriched['News_Sentiment'] = 0.0
    enriched['Analyst_Score']  = analyst
    enriched.iloc[-n:, enriched.columns.get_loc('News_Sentiment')] = combined * decay

    macro = fetch_macro_features(CONFIG['start_date'], CONFIG['end_date'])
    if not macro.empty:
        enriched = enriched.join(macro, how='left')
        for col in macro.columns: enriched[col] = enriched[col].ffill().bfill()

    report = {
        'ticker': ticker, 'run_timestamp': datetime.now().isoformat(),
        'news_sentiment': round(ns,4), 'alt_news_sentiment': round(als,4),
        'combined_sentiment': round(combined,4), 'analyst_score': round(analyst,4),
        'yahoo_count': len(yahoo), 'alt_news_count': len(alt),
    }
    print(f'✅ Agent 1 done. New cols: {[c for c in enriched.columns if c not in df.columns]}')
    return enriched, report

df_raw_enriched, sentiment_report = run_data_agent(df_raw_enriched, CONFIG['ticker'])
print(f'\nSentiment:\n{json.dumps(sentiment_report, indent=2)}')


🧠 Loading FinBERT...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ FinBERT ready

🤖 AGENT 1: AAPL
✅ Agent 1 done. New cols: ['News_Sentiment', 'Analyst_Score', 'Fed_Funds_Rate', 'Yield_Curve', 'VIX_FRED', 'CPI']

Sentiment:
{
  "ticker": "AAPL",
  "run_timestamp": "2026-06-19T11:28:24.384617",
  "news_sentiment": 0.3947,
  "alt_news_sentiment": 0.1338,
  "combined_sentiment": 0.2642,
  "analyst_score": 0.0,
  "yahoo_count": 20,
  "alt_news_count": 25
}


In [8]:
print("Rows downloaded:", len(df_raw_enriched))
print("Start Date setting:", CONFIG['start_date'])

Rows downloaded: 2882
Start Date setting: 2015-01-01


## ⚙️ Step 3 — Feature Engineering

In [9]:
def engineer_features(df):
    data = df.copy()

    # Check if there is enough data for feature engineering
    # The largest window is 200 for SMA_200.
    min_rows_needed = 200
    if len(data) < min_rows_needed:
        raise ValueError(
            f"Insufficient data for feature engineering. Expected at least "
            f"{min_rows_needed} rows, but got {len(data)}. "
            f"Please ensure 'download_stock_data' fetches enough historical data."
        )

    close = data['Close'].squeeze(); high=data['High'].squeeze()
    low=data['Low'].squeeze();       vol=data['Volume'].squeeze()
    data['SMA_20']  = SMAIndicator(close,20).sma_indicator()
    data['SMA_50']  = SMAIndicator(close,50).sma_indicator()
    data['SMA_200'] = SMAIndicator(close,200).sma_indicator()
    data['EMA_12']  = EMAIndicator(close,12).ema_indicator()
    data['EMA_26']  = EMAIndicator(close,26).ema_indicator()
    macd=MACD(close); data['MACD']=macd.macd(); data['MACD_Signal']=macd.macd_signal(); data['MACD_Hist']=macd.macd_diff()
    data['RSI_14'] = RSIIndicator(close,14).rsi()
    stoch=StochasticOscillator(high,low,close); data['Stoch_K']=stoch.stoch(); data['Stoch_D']=stoch.stoch_signal()
    bb=BollingerBands(close,20,2)
    data['BB_Upper']=bb.bollinger_hband(); data['BB_Middle']=bb.bollinger_mavg()
    data['BB_Lower']=bb.bollinger_lband(); data['BB_Width']=bb.bollinger_wband(); data['BB_Pct']=bb.bollinger_pband()
    data['ATR_14']         = AverageTrueRange(high,low,close,14).average_true_range()
    data['Daily_Return']   = close.pct_change()
    data['Log_Return']     = np.log(close/close.shift(1))
    data['Rolling_Std_20'] = close.rolling(20).std()
    data['OBV']            = OnBalanceVolumeIndicator(close,vol).on_balance_volume()
    data['Volume_SMA_20']  = vol.rolling(20).mean()
    data['Volume_Ratio']   = vol/data['Volume_SMA_20']
    for lag in [1,2,3,5,10]: data[f'Return_Lag_{lag}']=np.log(close/close.shift(lag))
    data['Rolling_Return_5']  = data['Log_Return'].rolling(5).mean()
    data['Rolling_Return_10'] = data['Log_Return'].rolling(10).mean()
    data['Rolling_Return_20'] = data['Log_Return'].rolling(10).mean()
    data['Price_vs_SMA20'] = (close-data['SMA_20'])/data['SMA_20']
    data['Price_vs_SMA50'] = (close-data['SMA_50'])/data['SMA_50']
    data['Day_of_Week']=data.index.dayofweek; data['Month']=data.index.month; data['Quarter']=data.index.quarter
    # Target = log-return N days ahead (avoids price scale issues)
    data['Target'] = np.log(close.shift(-CONFIG['forecast_days'])/close)
    data['Intraday_Return'] = (data['Close'] - data['Open']) / data['Open']
    data.dropna(inplace=True)
    print(f'✅ Features: {data.shape[1]} cols, {len(data)} rows')
    return data

df_features = engineer_features(df_raw_enriched)

# ── Bug 9 Fix: Sentiment/macro EXCLUDED from training features ─────────────
# Sentiment is sparse (only last 30 rows); macro is forward-filled constants.
# Both patterns confuse tree models. Kept in df_features for LLM report only.
EXCLUDE = [
    'Open','High','Low','Close','Volume','Target',
    'SPY_Close','QQQ_Close','XLK_Close','VIX_Close',
    'NSEI_Close','BSESN_Close',
    'News_Sentiment','Analyst_Score',
    'Fed_Funds_Rate','Yield_Curve','VIX_FRED','CPI',
]
feature_cols = [c for c in df_features.columns if c not in EXCLUDE]
print(f'Features ({len(feature_cols)}): {feature_cols}')

✅ Features: 61 cols, 2681 rows
Features (45): ['SPY_Return', 'SPY_Lag1', 'QQQ_Return', 'QQQ_Lag1', 'XLK_Return', 'XLK_Lag1', 'VIX_Return', 'VIX_Lag1', 'SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'MACD_Hist', 'RSI_14', 'Stoch_K', 'Stoch_D', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'BB_Width', 'BB_Pct', 'ATR_14', 'Daily_Return', 'Log_Return', 'Rolling_Std_20', 'OBV', 'Volume_SMA_20', 'Volume_Ratio', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3', 'Return_Lag_5', 'Return_Lag_10', 'Rolling_Return_5', 'Rolling_Return_10', 'Rolling_Return_20', 'Price_vs_SMA20', 'Price_vs_SMA50', 'Day_of_Week', 'Month', 'Quarter', 'Intraday_Return']


## ✂️ Step 4 — Split & Evaluation Framework

In [10]:
def split_data(df, feature_cols, test_size):
    X=df[feature_cols].values; y=df['Target'].values
    idx=int(len(X)*(1-test_size))
    X_train,X_test=X[:idx],X[idx:]
    y_train,y_test=y[:idx],y[idx:]
    train_dates=df.index[:idx]; test_dates=df.index[idx:]
    scaler=StandardScaler()
    X_train_sc=scaler.fit_transform(X_train)
    X_test_sc=scaler.transform(X_test)
    print(f'Train: {len(X_train):,} | {train_dates[0].date()} → {train_dates[-1].date()}')
    print(f'Test : {len(X_test):,}  | {test_dates[0].date()} → {test_dates[-1].date()}')
    return X_train_sc, X_test_sc, y_train, y_test, scaler, test_dates

X_train,X_test,y_train,y_test,scaler,test_dates = split_data(df_features,feature_cols,CONFIG['test_size'])

# ── Fixed evaluate_model — replace wherever it's defined in your notebook ──
def evaluate_model(y_true, y_pred, name='Model'):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    # Remove NaN pairs (last row of each WF window has NaN Target)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]

    if len(y_true) == 0:
        return {'Model': name, 'RMSE': np.nan, 'MAE': np.nan,
                'MAPE (%)': np.nan, 'R2': np.nan,
                'Directional Acc %': np.nan, 'P-Value': np.nan,
                'Sig (p<0.05)': '❌', 'RMSE_CI_95': '[nan,nan]'}

    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)

    # MAPE — guard against near-zero true values
    nonzero = np.abs(y_true) > 1e-8
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero])
                           / y_true[nonzero])) * 100 if nonzero.any() else np.nan

    # DirAcc — use boolean > 0, NOT np.sign()
    # np.sign(0.0) = 0.0, which equals neither +1 nor -1
    # causing 0% DirAcc whenever XGBoost predicts near-zero in calm markets
    dir_acc = np.mean((y_pred > 0) == (y_true > 0)) * 100

    # Bootstrap RMSE CI
    rng = np.random.default_rng(42)
    boot = [np.sqrt(mean_squared_error(
                y_true[idx := rng.integers(0, len(y_true), len(y_true))],
                y_pred[idx])) for _ in range(1000)]
    ci = f'[{np.percentile(boot,2.5):.6f},{np.percentile(boot,97.5):.6f}]'

    # Wilcoxon test vs naive baseline (predict zero return)
    naive  = np.zeros_like(y_true)
    errors_model = np.abs(y_true - y_pred)
    errors_naive = np.abs(y_true - naive)
    try:
        from scipy.stats import wilcoxon
        _, pval = wilcoxon(errors_model, errors_naive)
    except Exception:
        pval = 1.0

    return {
        'Model'             : name,
        'RMSE'              : round(rmse,  6),
        'RMSE_CI_95'        : ci,
        'MAE'               : round(mae,   6),
        'MAPE (%)'          : round(mape,  6) if not np.isnan(mape) else np.nan,
        'R2'                : round(r2,    4),
        'Directional Acc %' : round(dir_acc, 2),
        'P-Value'           : round(pval,  4),
        'Sig (p<0.05)'      : '✅ Yes' if pval < 0.05 else '❌ No',
    }

def plot_confusion_matrix(y_true, y_pred, model_name):
    true_dir=(np.diff(y_true)>0).astype(int)
    pred_dir=(np.diff(y_pred)>0).astype(int)
    cm=confusion_matrix(true_dir,pred_dir)
    fig,ax=plt.subplots(figsize=(5,4))
    # FIX: cmap (not colormap) is the correct kwarg for ConfusionMatrixDisplay.plot
    ConfusionMatrixDisplay(cm,display_labels=['DOWN','UP']).plot(ax=ax,cmap='Blues',values_format='d')
    ax.set_title(f'{model_name} — Direction Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f"confusion_{model_name.replace(' ','_')}.png",dpi=120,bbox_inches='tight')
    plt.show()
    tn,fp,fn,tp=cm.ravel()
    print(f'  Precision: {tp/(tp+fp+1e-8):.2%} | Recall: {tp/(tp+fn+1e-8):.2%}')

print('✅ Eval functions ready')


Train: 2,144 | 2015-10-16 → 2024-04-24
Test : 537  | 2024-04-25 → 2026-06-16
✅ Eval functions ready


## 🎯 Step 5 — Optuna Hyperparameter Tuning

In [11]:
tscv = TimeSeriesSplit(n_splits=CONFIG['n_folds'])

def xgb_objective(trial):
    p={'n_estimators':trial.suggest_int('n_estimators',200,800),
       'max_depth':trial.suggest_int('max_depth',3,9),
       'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
       'subsample':trial.suggest_float('subsample',0.6,1.0),
       'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
       'reg_alpha':trial.suggest_float('reg_alpha',1e-4,5.0,log=True),
       'reg_lambda':trial.suggest_float('reg_lambda',1e-4,5.0,log=True),
       'random_state':42,'n_jobs':-1}
    return np.mean([np.sqrt(mean_squared_error(y_train[te],
        xgb.XGBRegressor(**p,verbosity=0).fit(X_train[tr],y_train[tr]).predict(X_train[te])))
        for tr,te in tscv.split(X_train)])

print('🔍 Optuna: XGBoost (40 trials)...')
xs=optuna.create_study(direction='minimize')
xs.optimize(xgb_objective,n_trials=40,show_progress_bar=True)
best_xgb_params={**xs.best_params,'random_state':42,'n_jobs':-1}
print(f'✅ Best XGBoost RMSE: {xs.best_value:.6f}')
print(f'   Params: {best_xgb_params}')

def lgb_objective(trial):
    p={'n_estimators':trial.suggest_int('n_estimators',200,800),
       'max_depth':trial.suggest_int('max_depth',3,9),
       'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
       'num_leaves':trial.suggest_int('num_leaves',20,100),
       'subsample':trial.suggest_float('subsample',0.6,1.0),
       'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
       'random_state':42,'n_jobs':-1,'verbose':-1}
    return np.mean([np.sqrt(mean_squared_error(y_train[te],
        lgb.LGBMRegressor(**p).fit(X_train[tr],y_train[tr]).predict(X_train[te])))
        for tr,te in tscv.split(X_train)])

print('\n🔍 Optuna: LightGBM (40 trials)...')
ls=optuna.create_study(direction='minimize')
ls.optimize(lgb_objective,n_trials=40,show_progress_bar=True)
best_lgb_params={**ls.best_params,'random_state':42,'n_jobs':-1,'verbose':-1}
print(f'✅ Best LightGBM RMSE: {ls.best_value:.6f}')


🔍 Optuna: XGBoost (40 trials)...


  0%|          | 0/40 [00:00<?, ?it/s]

✅ Best XGBoost RMSE: 0.018294
   Params: {'n_estimators': 410, 'max_depth': 4, 'learning_rate': 0.034134395547931004, 'subsample': 0.6592053485846714, 'colsample_bytree': 0.6628263164740844, 'reg_alpha': 1.0086166905830896, 'reg_lambda': 0.1624223093542315, 'random_state': 42, 'n_jobs': -1}

🔍 Optuna: LightGBM (40 trials)...


  0%|          | 0/40 [00:00<?, ?it/s]

✅ Best LightGBM RMSE: 0.019198


## 🏋️ Step 6 — Train All Models

In [12]:
predictions, metrics_list = {}, []

def train_eval(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr); pred=model.predict(Xte)
    predictions[name]=pred
    m=evaluate_model(yte,pred,name); metrics_list.append(m)
    print(f"  {name}: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}% p={m['P-Value']} {m['Sig (p<0.05)']}")
    return pred

print('🔵 Linear Regression...'); y_pred_lr    = train_eval('Linear Regression',LinearRegression(),X_train,X_test,y_train,y_test)
print('🔵 Ridge Regression...');  y_pred_ridge = train_eval('Ridge Regression',Ridge(alpha=1.0),X_train,X_test,y_train,y_test)

print('🟠 XGBoost...')
xgb_model=xgb.XGBRegressor(**best_xgb_params,verbosity=0)
xgb_model.fit(X_train,y_train,eval_set=[(X_test,y_test)],verbose=False)
y_pred_xgb=xgb_model.predict(X_test)
predictions['XGBoost']=y_pred_xgb
m=evaluate_model(y_test,y_pred_xgb,'XGBoost'); metrics_list.append(m)
print(f"  XGBoost: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}% {m['Sig (p<0.05)']}")
xgb_model.save_model('xgboost_stock_model.json')

print('🟡 LightGBM...')
lgb_model=lgb.LGBMRegressor(**best_lgb_params)
lgb_model.fit(X_train,y_train)
y_pred_lgb=lgb_model.predict(X_test)
predictions['LightGBM']=y_pred_lgb
m=evaluate_model(y_test,y_pred_lgb,'LightGBM'); metrics_list.append(m)
print(f"  LightGBM: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}% {m['Sig (p<0.05)']}")
joblib.dump(lgb_model,'lightgbm_stock_model.pkl')

print('🟢 Random Forest...')
rf_model=RandomForestRegressor(n_estimators=300,max_depth=15,min_samples_split=5,
    min_samples_leaf=2,max_features='sqrt',random_state=CONFIG['random_state'],n_jobs=-1)
y_pred_rf=train_eval('Random Forest',rf_model,X_train,X_test,y_train,y_test)
joblib.dump(rf_model,'random_forest_stock_model.pkl')

# Weighted ensemble
y_pred_ensemble=y_pred_xgb*0.4+y_pred_lgb*0.4+y_pred_rf*0.2
predictions['Ensemble']=y_pred_ensemble
m=evaluate_model(y_test,y_pred_ensemble,'Ensemble'); metrics_list.append(m)
print(f"  Ensemble: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}% {m['Sig (p<0.05)']}")
print('✅ All tree models trained.')


🔵 Linear Regression...
  Linear Regression: RMSE=0.017685 DirAcc=50.09% p=0.0051 ✅ Yes
🔵 Ridge Regression...
  Ridge Regression: RMSE=0.017726 DirAcc=49.72% p=0.0042 ✅ Yes
🟠 XGBoost...
  XGBoost: RMSE=0.017605 DirAcc=51.21% ❌ No
🟡 LightGBM...
  LightGBM: RMSE=0.018858 DirAcc=46.74% ✅ Yes
🟢 Random Forest...
  Random Forest: RMSE=0.019411 DirAcc=46.74% p=0.0 ✅ Yes
  Ensemble: RMSE=0.018118 DirAcc=46.74% ✅ Yes
✅ All tree models trained.


## **Intraday Data Training**

In [13]:
def retrain_on_intraday(ticker, base_df_features, feature_cols, scaler):
    """
    Downloads today's intraday 5-min bars, converts them to features,
    appends to training data, and fine-tunes the XGBoost model on the
    combined dataset. Takes ~30 seconds. Run once per day at market open.

    This is what separates a static model from a live-aware one.
    The model now sees today's morning price action before predicting close.
    """
    print(f"\n🔄 Intraday fine-tune for {ticker}...")

    try:
        # ── Download today's 5-min bars ───────────────────────────────
        df_intra = yf.download(
            ticker, period='1d', interval='5m',
            auto_adjust=True, progress=False
        )
        if isinstance(df_intra.columns, pd.MultiIndex):
            df_intra.columns = df_intra.columns.get_level_values(0)
        df_intra = df_intra.dropna()

        if df_intra.empty or len(df_intra) < 20:
            print("  ⚠️  Not enough intraday bars — skipping fine-tune")
            return xgb_model, scaler

        # ── Engineer features on intraday bars ───────────────────────
        # Need market context columns — add zeros for now (they won't exist
        # in intraday data, but the model handles zero-fill)
        df_intra_feat = engineer_features(df_intra)
        common_cols   = [c for c in feature_cols if c in df_intra_feat.columns]

        # Build feature matrix — zero-fill missing cols
        X_intra = np.zeros((len(df_intra_feat), len(feature_cols)))
        for i, col in enumerate(feature_cols):
            if col in df_intra_feat.columns:
                X_intra[:, i] = df_intra_feat[col].values

        y_intra = df_intra_feat['Target'].values
        if len(y_intra) == 0:
            print("  ⚠️  No target rows after feature engineering — skipping")
            return xgb_model, scaler

        # ── Scale with existing scaler (do NOT refit — that leaks test data) ──
        X_intra_sc = scaler.transform(X_intra)

        # ── Fine-tune: continue training XGBoost on intraday data ────
        # xgb_model.fit with xgb_model as base (warm start via set_params)
        xgb_model_live = xgb.XGBRegressor(**best_xgb_params, verbosity=0)
        xgb_model_live.fit(
            X_intra_sc, y_intra,
            xgb_model=xgb_model.get_booster(),  # start from existing model
            verbose=False
        )

        print(f"  ✅ Fine-tuned on {len(df_intra_feat)} intraday bars")
        print(f"  Intraday price range: {df_intra['Close'].min():.2f} – "
              f"{df_intra['Close'].max():.2f}")
        return xgb_model_live, scaler

    except Exception as e:
        print(f"  ⚠️  Fine-tune failed: {e} — using original model")
        return xgb_model, scaler


# Run this once at the start of each trading day
xgb_model_live, _ = retrain_on_intraday(
    CONFIG['ticker'], df_features, feature_cols, scaler
)

# From here on, use xgb_model_live in predict_tomorrow instead of xgb_model
# The Gradio predict button already calls predict_tomorrow which uses xgb_model —
# replace that reference:
xgb_model = xgb_model_live   # swap in the live-aware version
print("✅ Live-aware model active")


🔄 Intraday fine-tune for AAPL...
  ⚠️  Fine-tune failed: Insufficient data for feature engineering. Expected at least 200 rows, but got 78. Please ensure 'download_stock_data' fetches enough historical data. — using original model
✅ Live-aware model active


## 🔴 Step 7 — LSTM

In [14]:
def create_sequences(X, y, lb):
    return (np.array([X[i-lb:i] for i in range(lb,len(X))]),
            np.array([y[i]        for i in range(lb,len(X))]))

lb = 30  # reduced from CONFIG['look_back']=60 for CPU speed
         # LSTM is weakest model (RMSE ~0.10), halving sequences
         # cuts training time ~50% with negligible quality impact

X_all_raw=df_features[feature_cols].values; y_all=df_features['Target'].values
sp_raw=int(len(X_all_raw)*(1-CONFIG['test_size']))

# Bug 1 Fix: Dedicated LSTM scaler fit ONLY on train portion
lstm_scaler=StandardScaler()
X_all_sc=np.vstack([
    lstm_scaler.fit_transform(X_all_raw[:sp_raw]),
    lstm_scaler.transform(X_all_raw[sp_raw:])
])

X_seq,y_seq=create_sequences(X_all_sc,y_all,lb)
sp=int(len(X_seq)*(1-CONFIG['test_size']))
X_lt,X_le=X_seq[:sp],X_seq[sp:]
y_lt,y_le=y_seq[:sp],y_seq[sp:]
lstm_test_dates=df_features.index[lb+sp:]

lstm_model=Sequential([
    LSTM(128,return_sequences=True,input_shape=(lb,X_lt.shape[2])),
    BatchNormalization(),Dropout(0.3),
    LSTM(64,return_sequences=True),BatchNormalization(),Dropout(0.2),
    LSTM(32),Dropout(0.2),Dense(16,activation='relu'),Dense(1)
])
lstm_model.compile(optimizer=Adam(1e-3),loss='huber',metrics=['mae'])
lstm_model.summary()
history=lstm_model.fit(X_lt,y_lt,validation_split=0.15,epochs=100,batch_size=32,shuffle=False,
    callbacks=[EarlyStopping(patience=5,restore_best_weights=True),  # reduced from 15 for CPU
               ReduceLROnPlateau(factor=0.5,patience=7,min_lr=1e-6)],verbose=1)
y_pred_lstm=lstm_model.predict(X_le).flatten()
predictions['LSTM']=(y_pred_lstm,lstm_test_dates)
m_lstm=evaluate_model(y_le,y_pred_lstm,'LSTM'); metrics_list.append(m_lstm)
print(f"LSTM: RMSE={m_lstm['RMSE']} DirAcc={m_lstm['Directional Acc %']}%")
lstm_model.save('lstm_stock_model.keras')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │        89,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 152,225 (594.63 KB)

 Trainable params: 151,841 (593.13 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 12s 99ms/step - loss: 0.0188 - mae: 0.1412 - val_loss: 0.0021 - val_mae: 0.0531 - learning_rate: 0.0010
Epoch 2/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - loss: 0.0040 - mae: 0.0683 - val_loss: 0.0015 - val_mae: 0.0436 - learning_rate: 0.0010
Epoch 3/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 73ms/step - loss: 0.0017 - mae: 0.0438 - val_loss: 0.0026 - val_mae: 0.0501 - learning_rate: 0.0010
Epoch 4/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 5s 80ms/step - loss: 0.0010 - mae: 0.0330 - val_loss: 0.0072 - val_mae: 0.0736 - learning_rate: 0.0010
Epoch 5/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - loss: 6.8759e-04 - mae: 0.0260 - val_loss: 0.0115 - val_mae: 0.0954 - learning_rate: 0.0010
Epoch 6/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 78ms/step - loss: 5.8245e-04 - mae: 0.0234 - val_loss: 0.0163 - val_mae: 0.1180 - learning_rate: 0.0010
Epoch 7/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - loss: 4.4039e-04 - mae: 0.0202 - val_loss: 0.0132 - val_mae: 0.1069 - learning_rate: 

## 🔄 Step 8 — Walk-Forward Validation

In [15]:
def walk_forward_validation(df, feature_cols, train_months=24, test_months=3):
    df = df.copy(); df['YM'] = df.index.to_period('M')
    periods = df['YM'].unique(); results = []

    for i in range(train_months, len(periods) - test_months, test_months):
        tr = df[df['YM'].isin(periods[i - train_months:i])]
        te = df[df['YM'].isin(periods[i:i + test_months])]

        # Drop NaN targets — last row of each window has NaN from shift(-1)
        tr = tr.dropna(subset=['Target'])
        te = te.dropna(subset=['Target'])

        if len(te) < 5: continue

        sc  = StandardScaler()
        Xtr = sc.fit_transform(tr[feature_cols])
        Xte = sc.transform(te[feature_cols])

        m   = xgb.XGBRegressor(**best_xgb_params, verbosity=0).fit(Xtr, tr['Target'])
        row = evaluate_model(te['Target'].values, m.predict(Xte), f'W{i}')
        row['test_start'] = str(periods[i]); results.append(row)

        print(f"  W{i}: {periods[i]} RMSE={row['RMSE']:.6f} "
              f"DirAcc={row['Directional Acc %']:.1f}%")

    wf = pd.DataFrame(results)
    print(f"\nWF: {len(wf)} windows | "
          f"Mean RMSE={wf['RMSE'].mean():.6f} ± {wf['RMSE'].std():.6f} | "
          f"Mean DirAcc={wf['Directional Acc %'].mean():.1f}% "
          f"± {wf['Directional Acc %'].std():.1f}%")
    return wf

wf_results = walk_forward_validation(df_features, feature_cols)


  W24: 2017-10 RMSE=0.011250 DirAcc=54.0%
  W27: 2018-01 RMSE=0.016772 DirAcc=44.3%
  W30: 2018-04 RMSE=0.013300 DirAcc=53.1%
  W33: 2018-07 RMSE=0.013060 DirAcc=65.1%
  W36: 2018-10 RMSE=0.026228 DirAcc=42.9%
  W39: 2019-01 RMSE=0.020911 DirAcc=65.6%
  W42: 2019-04 RMSE=0.016285 DirAcc=49.2%
  W45: 2019-07 RMSE=0.016550 DirAcc=54.7%
  W48: 2019-10 RMSE=0.012011 DirAcc=64.1%
  W51: 2020-01 RMSE=0.042246 DirAcc=43.5%
  W54: 2020-04 RMSE=0.021141 DirAcc=63.5%
  W57: 2020-07 RMSE=0.027917 DirAcc=59.4%
  W60: 2020-10 RMSE=0.021684 DirAcc=48.4%
  W63: 2021-01 RMSE=0.020742 DirAcc=47.5%
  W66: 2021-04 RMSE=0.013135 DirAcc=54.0%
  W69: 2021-07 RMSE=0.013016 DirAcc=54.7%
  W72: 2021-10 RMSE=0.015227 DirAcc=57.8%
  W75: 2022-01 RMSE=0.018735 DirAcc=45.2%
  W78: 2022-04 RMSE=0.026294 DirAcc=48.4%
  W81: 2022-07 RMSE=0.019447 DirAcc=48.4%
  W84: 2022-10 RMSE=0.025212 DirAcc=44.4%
  W87: 2023-01 RMSE=0.014905 DirAcc=53.2%
  W90: 2023-04 RMSE=0.011541 DirAcc=50.0%
  W93: 2023-07 RMSE=0.013713 DirAc

## 🔍 Step 9 — SHAP

In [16]:
explainer=shap.TreeExplainer(xgb_model)
shap_values=explainer.shap_values(X_test)
plt.figure(figsize=(10,8))
shap.summary_plot(shap_values,X_test,feature_names=feature_cols,plot_type='bar',show=False)
plt.title('SHAP Feature Importance'); plt.tight_layout()
plt.savefig('shap_summary.png',dpi=150,bbox_inches='tight'); plt.show()
plt.figure(figsize=(10,8))
shap.summary_plot(shap_values,X_test,feature_names=feature_cols,show=False)
plt.tight_layout(); plt.savefig('shap_beeswarm.png',dpi=150,bbox_inches='tight'); plt.show()
shap.waterfall_plot(shap.Explanation(values=shap_values[-1],base_values=explainer.expected_value,
    data=X_test[-1],feature_names=feature_cols),show=True)
print('✅ SHAP done')


✅ SHAP done


## 🎯 Step 10 — Confusion Matrices

In [17]:
for name,pred in predictions.items():
    if name=='LSTM': plot_confusion_matrix(y_le,pred[0],'LSTM')
    else:            plot_confusion_matrix(y_test,pred,name)


  Precision: 53.64% | Recall: 54.69%
  Precision: 53.79% | Recall: 55.47%
  Precision: 46.47% | Recall: 48.83%
  Precision: 42.75% | Recall: 44.92%
  Precision: 47.35% | Recall: 48.83%
  Precision: 46.24% | Recall: 48.05%
  Precision: 48.81% | Recall: 48.81%


## 📅 Step 11 — Multi-Step Forecasting

In [18]:
def multistep_forecast(df,feature_cols,scaler,horizons=(1,3,5,10)):
    close=df['Close'].squeeze()
    X_raw = df[feature_cols].values
    sp    = int(len(X_raw) * (1 - CONFIG['test_size']))
    Xtr   = scaler.transform(X_raw[:sp])
    Xte   = scaler.transform(X_raw[sp:])
    res={'Date':df.index[sp:],'Actual_T1':close.values[sp:]}
    print('\n📅 Multi-step forecast:')
    for h in horizons:
        t = np.log(close.shift(-h) / close).values; ytr,yte=t[:sp],t[sp:]
        v=~np.isnan(ytr)
        m=xgb.XGBRegressor(**best_xgb_params,verbosity=0).fit(Xtr[v],ytr[v])
        vt=~np.isnan(yte); ph=np.full(len(Xte),np.nan); ph[vt]=m.predict(Xte[vt])
        res[f'Pred_T{h}']=ph
        print(f'  T+{h:2d}: RMSE={np.sqrt(mean_squared_error(yte[vt],ph[vt])):.6f}')
    df_r=pd.DataFrame(res).set_index('Date')
    fig=go.Figure()
    fig.add_trace(go.Scatter(x=df_r.index,y=df_r['Actual_T1'],name='Actual',line=dict(color='white',width=2)))
    for col,color in zip([f'Pred_T{h}' for h in horizons],['#2196F3','#4CAF50','#FF9800','#F44336']):
        fig.add_trace(go.Scatter(x=df_r.index,y=df_r[col],name=col,line=dict(color=color,dash='dash')))
    fig.update_layout(template='plotly_dark',title='Multi-Step Forecast',height=450)
    fig.write_html('multistep_forecast.html'); fig.show()
    return df_r

multistep_df=multistep_forecast(df_features,feature_cols,scaler,CONFIG['multistep_days'])



📅 Multi-step forecast:
  T+ 1: RMSE=0.017615
  T+ 3: RMSE=0.032943
  T+ 5: RMSE=0.042769
  T+10: RMSE=0.060719


## 📊 Step 12 — Model Performance Summary

In [28]:
metrics_df=pd.DataFrame(metrics_list).set_index('Model')
print('\nMODEL PERFORMANCE SUMMARY\n'+'='*90)
print(metrics_df[['RMSE', 'RMSE_CI_95', 'MAE', 'R2',
                   'Directional Acc %', 'P-Value', 'Sig (p<0.05)']].to_string())

# MAPE excluded from all tickers:
# Target = log-returns (values ~0.001–0.02).
# Dividing by near-zero denominators produces 100x–3000x% errors
# regardless of stock. MAPE is only meaningful for price-level targets.
# Use RMSE + Directional Accuracy as primary metrics instead.
print('='*90)
print(f"Best RMSE: {metrics_df['RMSE'].idxmin()} | Best DirAcc: {metrics_df['Directional Acc %'].idxmax()}")

# ── AAPL Price Comparison Cell ─────────────────────────────────────────────
# AAPL current price on NASDAQ as of Jun 18 2026 = ~$295.95
# The model predicts LOG-RETURNS, not prices.
# Ensemble log-return * 100 ≈ predicted % change for small moves.
# To convert to price: predicted_price = current_close * exp(log_return_pred)
#
# HOW TO INTERPRET:
#   If today's close = $295.95 and Ensemble predicts log_return = +0.003
#   then tomorrow's predicted price = 295.95 * exp(0.003) ≈ $296.84 (+$0.89, +0.30%)
#
# The RMSE in log-return space (~0.010–0.015) translates to:
#   Dollar RMSE ≈ 295.95 * 0.012 ≈ $3.55 per day
#   That is a NORMAL range for a day-ahead stock predictor.
#
# DIRECTIONAL ACCURACY matters more than RMSE:
#   >55% = genuinely predictive
#   50–55% = marginal
#   <50% = worse than random

fig,axes=plt.subplots(1,3,figsize=(18,5))
fig.suptitle(f"{CONFIG['ticker']} Model Comparison",fontsize=14,fontweight='bold')
palette=['#2196F3','#9C27B0','#FF9800','#4CAF50','#F44336','#00BCD4','#FF5722']
for ax,metric in zip(axes,['RMSE','MAPE (%)','Directional Acc %']):
    vals=metrics_df[metric].sort_values(ascending=(metric!='Directional Acc %'))
    bars=ax.bar(vals.index,vals.values,color=palette[:len(vals)])
    ax.set_title(metric,fontsize=11)
    ax.set_xticklabels(vals.index,rotation=20,ha='right',fontsize=8)
    for bar,val in zip(bars,vals.values):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.001*max(vals),
                f'{val:.4f}',ha='center',va='bottom',fontsize=7)
    ax.grid(axis='y',alpha=0.3); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.savefig('model_comparison.png',dpi=150,bbox_inches='tight'); plt.show()



MODEL PERFORMANCE SUMMARY
                       RMSE           RMSE_CI_95       MAE       R2  Directional Acc %  P-Value Sig (p<0.05)
Model                                                                                                       
Linear Regression  0.017685  [0.015502,0.019968]  0.012100  -0.0134              50.09   0.0051        ✅ Yes
Ridge Regression   0.017726  [0.015540,0.019993]  0.012110  -0.0181              49.72   0.0042        ✅ Yes
XGBoost            0.017605  [0.015273,0.020237]  0.011820  -0.0042              51.21   0.5146         ❌ No
LightGBM           0.018858  [0.016825,0.021146]  0.013669  -0.1523              46.74   0.0000        ✅ Yes
Random Forest      0.019411  [0.017467,0.021608]  0.014211  -0.2209              46.74   0.0000        ✅ Yes
Ensemble           0.018118  [0.016006,0.020561]  0.012751  -0.0636              46.74   0.0000        ✅ Yes
LSTM               0.080485  [0.076421,0.084150]  0.066892 -20.3597              48.02   0.0000      

## 🔮 Step 13 — Live Inference (predict_tomorrow)

In [20]:
def predict_tomorrow(ticker):
    """
    v3.1 (Robust) — injects live intraday data into a dedicated feature
    to avoid corrupting the daily time-series lags.
    """
    end   = datetime.today().strftime('%Y-%m-%d')
    start = (datetime.today() - timedelta(days=600)).strftime('%Y-%m-%d')

    # ── Download historical daily data (for feature warmup) ───────────
    df_hist = download_stock_data(ticker, start, end)
    df_hist = add_market_context(df_hist)

    df_hist = engineer_features(df_hist)

    # ── Get live price from agent ──────────────────────────────────────
    live_info  = get_live_price(ticker)
    live_price = live_info['price'] if (live_info.get('price', 0) > 0
                                        and not live_info.get('error')) else None

    # ── Inject live data into last row if available ────────────────────
    if live_price and live_price > 0:
        last_close = float(df_hist['Close'].iloc[-1])
        today_open = float(df_hist['Open'].iloc[-1])

        # Calculate live intraday return from today's open
        live_intra_ret = (live_price - today_open) / today_open if today_open > 0 else 0

        # ONLY update the dedicated Intraday feature and SMA distances.
        # DO NOT touch Return_Lag_1 or Daily_Return to keep the daily time-series pure.
        if 'Intraday_Return' in df_hist.columns:
            df_hist.iloc[-1, df_hist.columns.get_loc('Intraday_Return')] = live_intra_ret

        if 'SMA_20' in df_hist.columns:
            df_hist.iloc[-1, df_hist.columns.get_loc('Price_vs_SMA20')] = (
                (live_price - float(df_hist['SMA_20'].iloc[-1])) / float(df_hist['SMA_20'].iloc[-1])
            )

        if 'SMA_50' in df_hist.columns:
            df_hist.iloc[-1, df_hist.columns.get_loc('Price_vs_SMA50')] = (
                (live_price - float(df_hist['SMA_50'].iloc[-1])) / float(df_hist['SMA_50'].iloc[-1])
            )

        print(f"  ✅ Live data injected cleanly: live=${live_price:.2f} (intraday momentum: {live_intra_ret*100:+.2f}%)")
        cp = live_price   # use live as base for price conversion
    else:
        cp = float(df_hist['Close'].iloc[-1])
        print(f"  ⚠️  No live data — using last close: {cp:.2f}")

    # ── Build feature vector ───────────────────────────────────────────
    lat_full = np.zeros((1, len(feature_cols)))
    missing  = []
    for i, col in enumerate(feature_cols):
        if col in df_hist.columns:
            lat_full[0, i] = float(df_hist[col].iloc[-1])
        else:
            missing.append(col)
    if missing:
        print(f"  ⚠️  {len(missing)} features zero-filled: {missing[:3]}...")

    ls     = scaler.transform(lat_full)
    lr_xgb = float(xgb_model.predict(ls)[0])
    lr_lgb = float(lgb_model.predict(ls)[0])
    lr_rf  = float(rf_model.predict(ls)[0])
    lr_ens = lr_xgb*0.4 + lr_lgb*0.4 + lr_rf*0.2

    px_xgb = round(cp * np.exp(lr_xgb), 2)
    px_lgb = round(cp * np.exp(lr_lgb), 2)
    px_rf  = round(cp * np.exp(lr_rf),  2)
    px_ens = round(cp * np.exp(lr_ens), 2)
    pct    = lr_ens * 100

    currency = live_info.get('currency', 'INR' if '.NS' in ticker else 'USD')
    sym      = '₹' if currency == 'INR' else '$'

    result = {
        'ticker'          : ticker,
        'current_price'   : round(cp, 2),
        'live_price'      : round(live_price or cp, 2),
        'last_data_date'  : str(df_hist.index[-1].date()),
        'currency'        : currency,
        'symbol'          : sym,
        'xgboost'         : px_xgb,
        'lightgbm'        : px_lgb,
        'random_forest'   : px_rf,
        'ensemble'        : px_ens,
        'pct_change'      : round(pct, 3),
        'signal'          : '📈 BULLISH' if lr_ens > 0 else '📉 BEARISH',
        'prediction_date' : (datetime.today() + timedelta(days=1)).strftime('%Y-%m-%d'),
        'live_injected'   : live_price is not None,
        'market_state'    : live_info.get('market_state', 'UNKNOWN'),
        'change_today'    : live_info.get('change', 0),
        'change_pct_today': live_info.get('change_pct', 0),
    }

    print(f"\n{'='*55}")
    print(f"  {ticker} | Live injected: {result['live_injected']}")
    print(f"  Live Price   : {sym}{result['live_price']:.2f} "
          f"({result['change_pct_today']:+.2f}% today)")
    print(f"  Ensemble     : {sym}{px_ens:.2f} ({pct:+.2f}%)")
    print(f"  Signal       : {result['signal']}")
    print(f"{'='*55}")
    return result

tomorrow = predict_tomorrow(CONFIG['ticker'])

📡 Downloading AAPL...
  ✅ 411 days | Close: 171.51 – 315.20
  ✅ SPY: 410 rows
  ✅ QQQ: 410 rows
  ✅ XLK: 410 rows
  ✅ ^VIX: 410 rows
✅ Features: 55 cols, 210 rows
  📡 AAPL: source=Finnhub price=USD 298.01
  ✅ Live data injected cleanly: live=$298.01 (intraday momentum: +0.93%)

  AAPL | Live injected: True
  Live Price   : $298.01 (+0.70% today)
  Ensemble     : $296.83 (-0.40%)
  Signal       : 📉 BEARISH


In [21]:
# pip install langgraph langchain-groq

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from typing import TypedDict

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=KEYS['groq'])

class TradingState(TypedDict):
    ticker: str
    ml_signal: dict        # your existing predict_tomorrow() output
    shap_features: list    # top SHAP features from your model
    news_sentiment: float  # your existing sentiment score
    technical_report: str
    sentiment_report: str
    bull_case: str
    bear_case: str
    final_decision: str

def technical_analyst(state):
    ml = state['ml_signal']
    prompt = f"""You are a technical analyst. The ML pipeline produced:
    - Ensemble prediction: {ml['symbol']}{ml['ensemble']} ({ml['pct_change']:+.2f}%)
    - Signal: {ml['signal']}
    - Top features driving this: {state['shap_features'][:5]}
    - Live price: {ml['symbol']}{ml['live_price']}
    Write a 3-bullet technical analysis."""
    report = llm.invoke(prompt).content
    return {**state, 'technical_report': report}

def sentiment_analyst(state):
    prompt = f"""You are a sentiment analyst for {state['ticker']}.
    News sentiment score: {state['news_sentiment']} (range -1 to +1)
    Write a 2-bullet sentiment summary."""
    report = llm.invoke(prompt).content
    return {**state, 'sentiment_report': report}

def bull_researcher(state):
    prompt = f"""You are a BULLISH researcher for {state['ticker']}.
    Technical: {state['technical_report']}
    Sentiment: {state['sentiment_report']}
    Make the strongest possible bull case in 3 points."""
    case = llm.invoke(prompt).content
    return {**state, 'bull_case': case}

def bear_researcher(state):
    prompt = f"""You are a BEARISH researcher for {state['ticker']}.
    Technical: {state['technical_report']}
    Sentiment: {state['sentiment_report']}
    Make the strongest possible bear case in 3 points."""
    case = llm.invoke(prompt).content
    return {**state, 'bear_case': case}

def portfolio_manager(state):
    prompt = f"""You are a portfolio manager for {state['ticker']}.
    Bull case: {state['bull_case']}
    Bear case: {state['bear_case']}
    ML ensemble says: {state['ml_signal']['signal']}
    ({state['ml_signal']['pct_change']:+.2f}% predicted)

    Give a final decision: BUY / SELL / HOLD
    With position size: FULL / HALF / QUARTER
    And a one paragraph rationale."""
    decision = llm.invoke(prompt).content
    return {**state, 'final_decision': decision}

# Build the graph
graph = StateGraph(TradingState)
graph.add_node("technical", technical_analyst)
graph.add_node("sentiment", sentiment_analyst)
graph.add_node("bull",      bull_researcher)
graph.add_node("bear",      bear_researcher)
graph.add_node("pm",        portfolio_manager)

graph.set_entry_point("technical")
graph.add_edge("technical", "sentiment")
graph.add_edge("sentiment", "bull")
graph.add_edge("sentiment", "bear")
graph.add_edge("bull",      "pm")
graph.add_edge("bear",      "pm")
graph.add_edge("pm",        END)

trading_graph = graph.compile()

In [22]:
!pip install langgraph langchain-groq -q

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=KEYS['groq'])

# ── Reducer: keeps the most recent non-empty value ─────────────────────────
def keep_latest(a, b):
    """
    If b is a real value (non-empty string, non-empty dict, non-None),
    use b. Otherwise keep a. This lets initial values flow in correctly
    AND lets nodes update their own output keys without conflict.
    """
    if b is None:
        return a
    if isinstance(b, str) and b.strip() == "":
        return a
    if isinstance(b, dict) and len(b) == 0:
        return a
    if isinstance(b, list) and len(b) == 0:
        return a
    return b

# ── State definition ────────────────────────────────────────────────────────
class TradingState(TypedDict):
    ticker:           Annotated[str,   keep_latest]
    ml_signal:        Annotated[dict,  keep_latest]
    shap_features:    Annotated[list,  keep_latest]
    news_sentiment:   Annotated[float, keep_latest]
    technical_report: Annotated[str,   keep_latest]
    sentiment_report: Annotated[str,   keep_latest]
    bull_case:        Annotated[str,   keep_latest]
    bear_case:        Annotated[str,   keep_latest]
    final_decision:   Annotated[str,   keep_latest]

# ── Agent nodes — each returns ONLY the key it writes ──────────────────────
# This is the real fix: returning {**state, 'key': value} causes every
# key to be "updated" simultaneously → InvalidUpdateError on parallel branches.
# Returning only {'key': value} tells LangGraph exactly what changed.

def technical_analyst(state: TradingState) -> dict:
    ml = state['ml_signal']
    sym        = ml.get('symbol', '$')
    ensemble   = ml.get('ensemble', 0.0)
    pct_change = ml.get('pct_change', 0.0)
    signal     = ml.get('signal', 'UNKNOWN')
    live_price = ml.get('live_price', 0.0)
    top_feats  = state.get('shap_features', [])[:5]

    prompt = f"""You are a quantitative technical analyst.
The ML pipeline produced these results for {state['ticker']}:
- Ensemble price forecast : {sym}{ensemble:.2f} ({pct_change:+.2f}%)
- Signal                  : {signal}
- Live price              : {sym}{live_price:.2f}
- Top 5 SHAP features     : {top_feats}

Write exactly 3 bullet points of technical analysis.
Be specific about what the numbers imply."""

    report = llm.invoke(prompt).content
    return {'technical_report': report}          # ← only write what changed


def sentiment_analyst(state: TradingState) -> dict:
    prompt = f"""You are a market sentiment analyst for {state['ticker']}.
News sentiment score: {state['news_sentiment']:.4f}  (scale: -1 very negative → +1 very positive)

Write exactly 2 bullet points interpreting this sentiment score
and what it implies for short-term price action."""

    report = llm.invoke(prompt).content
    return {'sentiment_report': report}          # ← only write what changed


def bull_researcher(state: TradingState) -> dict:
    prompt = f"""You are a BULLISH equity researcher for {state['ticker']}.
You have access to these reports:

TECHNICAL ANALYSIS:
{state['technical_report']}

SENTIMENT ANALYSIS:
{state['sentiment_report']}

Make the strongest possible bull case in exactly 3 points.
Focus on upside catalysts and why the stock should rise tomorrow."""

    case = llm.invoke(prompt).content
    return {'bull_case': case}                   # ← only write what changed


def bear_researcher(state: TradingState) -> dict:
    prompt = f"""You are a BEARISH equity researcher for {state['ticker']}.
You have access to these reports:

TECHNICAL ANALYSIS:
{state['technical_report']}

SENTIMENT ANALYSIS:
{state['sentiment_report']}

Make the strongest possible bear case in exactly 3 points.
Focus on downside risks and why the stock might fall or underperform tomorrow."""

    case = llm.invoke(prompt).content
    return {'bear_case': case}                   # ← only write what changed


def portfolio_manager(state: TradingState) -> dict:
    ml   = state['ml_signal']
    sym  = ml.get('symbol', '$')
    sig  = ml.get('signal', 'UNKNOWN')
    pct  = ml.get('pct_change', 0.0)
    ens  = ml.get('ensemble', 0.0)

    prompt = f"""You are the Portfolio Manager making the final trading decision for {state['ticker']}.

ML ENSEMBLE FORECAST : {sym}{ens:.2f} ({pct:+.2f}%) — {sig}

BULL RESEARCHER SAYS:
{state['bull_case']}

BEAR RESEARCHER SAYS:
{state['bear_case']}

Synthesize all of the above and give:
1. Decision    : BUY / SELL / HOLD
2. Position    : FULL / HALF / QUARTER
3. Rationale   : One clear paragraph explaining the decision,
                 acknowledging the strongest point from each side."""

    decision = llm.invoke(prompt).content
    return {'final_decision': decision}          # ← only write what changed


# ── Build graph ─────────────────────────────────────────────────────────────
graph = StateGraph(TradingState)

graph.add_node("technical", technical_analyst)
graph.add_node("sentiment", sentiment_analyst)
graph.add_node("bull",      bull_researcher)
graph.add_node("bear",      bear_researcher)
graph.add_node("pm",        portfolio_manager)

graph.set_entry_point("technical")
graph.add_edge("technical", "sentiment")
graph.add_edge("sentiment", "bull")
graph.add_edge("sentiment", "bear")
graph.add_edge("bull",      "pm")
graph.add_edge("bear",      "pm")
graph.add_edge("pm",        END)

trading_graph = graph.compile()
print("✅ Trading graph compiled successfully")

✅ Trading graph compiled successfully


In [23]:
def technical_analyst(state: TradingState) -> dict:
    print("\n🔵 TECHNICAL ANALYST running...")
    # ... rest of function
    print(f"✅ Technical report generated ({len(report)} chars)")
    return {'technical_report': report}

def sentiment_analyst(state: TradingState) -> dict:
    print("\n🟡 SENTIMENT ANALYST running...")
    # ...
    print(f"✅ Sentiment report done")
    return {'sentiment_report': report}

def bull_researcher(state: TradingState) -> dict:
    print("\n🟢 BULL RESEARCHER running...")
    # ...
    print(f"✅ Bull case built")
    return {'bull_case': case}

def bear_researcher(state: TradingState) -> dict:
    print("\n🔴 BEAR RESEARCHER running...")
    # ...
    print(f"✅ Bear case built")
    return {'bear_case': case}

def portfolio_manager(state: TradingState) -> dict:
    print("\n🏦 PORTFOLIO MANAGER making final decision...")
    # ...
    print(f"✅ Decision reached")
    return {'final_decision': decision}

## 💰 Step 14 — Backtest (Fixed · Risk-Managed)

In [24]:
def backtest_strategy(prices_true, y_pred, dates, model_name,
                      cap=10000.0, tc=0.001, sl=0.05, tp=0.10):
    """
    Fixed backtest:
    - prices_true : actual closing prices (NOT log-returns)
    - y_pred      : predicted log-returns (signal: >0 = LONG, <=0 = FLAT)
    - No lookahead: signal at day i → trade executes at open of day i+1
    - Stop-loss 5%, take-profit 10%, transaction cost 0.1%
    """
    capital, pos, entry = cap, 0.0, 0.0
    eq, bh, trades = [], [], []
    prices = np.array(prices_true, dtype=float)
    preds  = np.array(y_pred,      dtype=float)

    # Validate inputs — catch the log-return / price confusion early
    if prices.mean() < 1.0:
        raise ValueError(
            f'prices_true looks like log-returns (mean={prices.mean():.4f}). '
            f'Pass actual closing prices, not y_test.')

    bhs = cap / prices[0]   # shares in buy-and-hold benchmark

    for i in range(len(prices) - 1):
        cp  = prices[i]
        sig = 1 if preds[i] > 0 else -1

        # Check stop-loss / take-profit before new signal
        if pos > 0 and entry > 0:
            r = (cp - entry) / entry
            if r <= -sl or r >= tp:
                capital = pos * cp * (1 - tc)
                trades.append({'type': 'STP/TP', 'ret': r})
                pos, entry = 0.0, 0.0

        # Open / close position
        if sig == 1 and pos == 0:
            pos    = (capital * (1 - tc)) / cp
            entry  = cp
            capital = 0.0
            trades.append({'type': 'BUY', 'ret': None})
        elif sig == -1 and pos > 0:
            r       = (cp - entry) / entry
            capital = pos * cp * (1 - tc)
            trades.append({'type': 'SELL', 'ret': r})
            pos, entry = 0.0, 0.0

        eq.append(capital + (pos * cp if pos > 0 else 0))
        bh.append(bhs * cp)

    # Close any open position at end
    if pos > 0:
        eq[-1] = pos * prices[-1] * (1 - tc)

    equity = np.array(eq); bha = np.array(bh)
    tr_ret = (equity[-1] - cap) / cap * 100
    bh_ret = (bha[-1]    - cap) / cap * 100

    dr     = np.diff(equity) / np.maximum(equity[:-1], 1e-10)
    sharpe = (dr.mean() / (dr.std() + 1e-10)) * np.sqrt(252)
    mdd    = ((equity - np.maximum.accumulate(equity)) /
               np.maximum.accumulate(equity)).min() * 100
    trets  = [t['ret'] for t in trades if t.get('ret') is not None]
    wr     = sum(r > 0 for r in trets) / max(len(trets), 1) * 100

    print(f'\n💰 {model_name}: Return={tr_ret:+.1f}% | B&H={bh_ret:+.1f}% | '
          f'Sharpe={sharpe:.2f} | MDD={mdd:.1f}% | '
          f'WinRate={wr:.1f}% | Trades={len(trets)}')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=list(dates[:len(equity)]), y=equity,
                              name='Strategy', line=dict(color='#4CAF50', width=2)))
    fig.add_trace(go.Scatter(x=list(dates[:len(bha)]),    y=bha,
                              name='Buy & Hold',
                              line=dict(color='#2196F3', width=2, dash='dash')))
    fig.add_hline(y=cap, line_dash='dot', line_color='white', opacity=0.5)
    fig.update_layout(
        template='plotly_dark',
        title=f'{model_name} vs Buy & Hold | Sharpe={sharpe:.2f}',
        height=420)
    fname = f"backtest_{model_name.replace(' ', '_')}.html"
    fig.write_html(fname); fig.show()

    return pd.DataFrame({
        'Date'    : dates[:len(equity)],
        'Strategy': equity,
        'BuyHold' : bha
    })


# ── Call with actual PRICES not y_test ────────────────────────────────────
# Extract closing prices aligned to the test period
test_prices = df_features['Close'].values[-len(y_test):]
test_dates  = df_features.index[-len(y_test):]

backtest_df = backtest_strategy(
    prices_true = test_prices,        # ← actual prices, NOT y_test
    y_pred      = y_pred_ensemble,    # ← log-return predictions (correct)
    dates       = test_dates,
    model_name  = 'Ensemble',
    cap         = CONFIG['initial_capital'],
    tc          = CONFIG['transaction_cost']
)


💰 Ensemble: Return=+22.8% | B&H=+76.2% | Sharpe=0.75 | MDD=-9.9% | WinRate=66.7% | Trades=18


## 📝 Step 15 — Groq LLM Report

In [25]:
def run_groq_agent(ticker,metrics_df,sentiment_report,tomorrow_pred,wf_summary):
    combined = sentiment_report.get('combined_sentiment', sentiment_report.get('alt_news_sentiment',0))
    s={
        'ticker':ticker,'best_model':metrics_df['RMSE'].idxmin(),
        'best_rmse':float(metrics_df['RMSE'].min()),
        'best_da':float(metrics_df['Directional Acc %'].max()),
        'sig':metrics_df[metrics_df['Sig (p<0.05)']=='✅ Yes'].index.tolist(),
        'wf_mu':float(wf_summary['RMSE'].mean()),
        'wf_sd':float(wf_summary['RMSE'].std()),
        'tm':tomorrow_pred,'sent':sentiment_report,
    }
    prompt=(
        f"Senior quant analyst reviewing ML stock prediction pipeline.\n"
        f"Stock:{s['ticker']} | Best:{s['best_model']}(RMSE:{s['best_rmse']:.6f}) | DirAcc:{s['best_da']:.1f}%\n"
        f"Significant models (p<0.05 vs naive): {s['sig']}\n"
        f"Walk-Forward RMSE: {s['wf_mu']:.6f} ± {s['wf_sd']:.6f}\n"
        f"Sentiment: news={s['sent']['news_sentiment']:.4f} combined={combined:.4f} analyst={s['sent']['analyst_score']:.4f}\n"
        f"Tomorrow: live={s['tm']['symbol']}{s['tm']['live_price']:.2f} "        f"ensemble={s['tm']['symbol']}{s['tm']['ensemble']:.2f} ({s['tm']['pct_change']:+.2f}%) "        f"signal={s['tm']['signal']}\n\n"
        f"Write 5-point analysis: model performance, sentiment signals, "        f"prediction confidence, key risks, one-line recommendation. <300 words, bullets."
    )
    if not KEYS.get('groq'):
        return (f"LOCAL REPORT — {ticker}\n"
                f"Best: {s['best_model']} RMSE={s['best_rmse']:.6f} DirAcc={s['best_da']:.1f}%\n"
                f"WF RMSE: {s['wf_mu']:.6f} ± {s['wf_sd']:.6f}\n"
                f"Signal: {s['tm']['signal']} | Ensemble: {s['tm']['symbol']}{s['tm']['ensemble']:.2f} ({s['tm']['pct_change']:+.2f}%)\n"
                f"Live: {s['tm']['symbol']}{s['tm']['live_price']:.2f} | Market: {s['tm']['market_state']}\n"
                f"Note: Add GROQ_API_KEY to Colab Secrets for Llama 3 narrative report.")
    try:
        from groq import Groq
        resp=Groq(api_key=KEYS['groq']).chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[{'role':'user','content':prompt}],
            max_tokens=500,temperature=0.3)
        return resp.choices[0].message.content
    except Exception as e: return f'Groq error: {e}'

llm_report=run_groq_agent(CONFIG['ticker'],metrics_df,sentiment_report,tomorrow,wf_results)
print('\n'+'='*60+'\nLLM REPORT\n'+'='*60+'\n'+llm_report)
with open('analysis_report.txt','w') as f: f.write(llm_report)



LLM REPORT
Here's a 5-point analysis of the ML stock prediction pipeline for AAPL:
* **Model Performance**: The best-performing model is XGBoost with an RMSE of 0.017605, and the walk-forward RMSE is 0.017913 ± 0.006854, indicating a relatively stable and accurate prediction.
* **Sentiment Signals**: Sentiment analysis shows a mixed signal, with news sentiment at 0.3947 and combined sentiment at 0.2642, while analyst sentiment is neutral (0.0000), which may indicate a potential disconnect between market expectations and analyst views.
* **Prediction Confidence**: The ensemble prediction for tomorrow is $296.83, which is 0.40% lower than the live price, with a bearish signal (📉), indicating a moderate level of confidence in the prediction.
* **Key Risks**: The key risks to the prediction include the relatively low directional accuracy (51.2%) of the best-performing model, and the potential for unexpected market events or news that could impact the stock price.
* **Recommendation**: Bas

## 🖥️ Step 16 — Gradio Dashboard (Real-Time Price Feed)

**What's new vs v2:**
- ⚡ **Auto-refreshing live ticker** — polls yfinance every 30 s, shows live price with arrow/color
- 🇮🇳 **Indian stocks** — enter `RELIANCE.NS`, `TCS.NS`, `INFY.NS`, `HDFCBANK.NS` etc.
- 📊 **Live price vs model prediction** side by side
- 🔄 **Manual refresh button** and **auto-refresh toggle**

**How auto-refresh works in Gradio:**  
`gr.Timer` fires `every=30` seconds and updates the live ticker row.  
The prediction itself only re-runs when you click **Predict** (it takes ~10 s to download + engineer features).


In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 16 — Gradio Dashboard v4.1
#
# FIXES vs v4:
#   1. Button label emoji rendered correctly (no HTML entities in gr.Button)
#   2. Volume=0 fallback — shows "N/A" instead of 0 for pre/post market
#   3. NSE currency guard — detects USD-priced .NS data and flags it
#   4. Cleaner layout — dark terminal aesthetic, monospace data, clear sections
#   5. Agent decision panel wired into Gradio (new tab)
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr
from datetime import datetime

_current_ticker = {"value": CONFIG["ticker"]}

# ─────────────────────────────────────────────────────────────────────────────
# CSS — dark terminal theme
# ─────────────────────────────────────────────────────────────────────────────
CUSTOM_CSS = """
/* ── global ── */
.gradio-container { background: #0a0a0f !important; }
.gr-block, .gr-box { background: transparent !important; }

/* ── section labels ── */
.section-label {
    font-family: 'JetBrains Mono', 'Fira Code', monospace;
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    color: #4a4a6a;
    margin: 18px 0 6px;
    padding-left: 2px;
}

/* ── price card ── */
.price-card {
    background: #0d0d1a;
    border: 1px solid #1e1e3a;
    border-radius: 12px;
    padding: 18px 20px;
    font-family: 'JetBrains Mono', monospace;
}
.price-card.up   { border-color: #1a3a1a; }
.price-card.down { border-color: #3a1a1a; }

/* ── prediction table ── */
.pred-card {
    background: #0d0d1a;
    border: 1px solid #1e1e3a;
    border-radius: 12px;
    padding: 18px 20px;
    font-family: 'JetBrains Mono', monospace;
}

/* ── watchlist ── */
.watch-card {
    background: #0a0a14;
    border: 1px solid #1a1a2e;
    border-radius: 12px;
    padding: 16px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 13px;
}

/* ── agent panel ── */
.agent-card {
    background: #0d0d1a;
    border: 1px solid #1e1e3a;
    border-radius: 12px;
    padding: 18px 20px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 13px;
    line-height: 1.7;
    white-space: pre-wrap;
}
"""

# ─────────────────────────────────────────────────────────────────────────────
# LIVE TICKER
# ─────────────────────────────────────────────────────────────────────────────
def refresh_live_ticker(ticker_input: str) -> str:
    ticker = (ticker_input or "").strip().upper() or CONFIG["ticker"]
    _current_ticker["value"] = ticker
    info = get_live_price(ticker)

    if info.get("error"):
        return f"""<div class='price-card' style='border-color:#3a1a1a;'>
  <div style='color:#666;font-size:11px;font-family:monospace;'>ERROR</div>
  <div style='color:#F44336;font-size:14px;margin-top:6px;'>{ticker}: {info["error"]}</div>
</div>"""

    sym       = "&#8377;" if info["currency"] == "INR" else "$"
    is_up     = info["change"] >= 0
    clr       = "#4CAF50" if is_up else "#F44336"
    arrow     = "&#9650;" if is_up else "&#9660;"
    sign      = "+" if is_up else ""
    card_cls  = "up" if is_up else "down"
    ms        = info.get("market_state", "")
    dot_color = "#4CAF50" if ms == "REGULAR" else ("#FFB300" if ("PRE" in ms or "POST" in ms) else "#666")
    vol       = f"{info['volume']:,}" if info.get('volume', 0) > 0 else "N/A"
    src       = info.get("source", "")
    ts        = info.get("timestamp", "")

    # Guard: .NS ticker returning suspiciously low price = USD leak
    currency_warning = ""
    if ticker.endswith(".NS") and info["currency"] == "INR" and info["price"] < 10:
        currency_warning = "<div style='color:#FFB300;font-size:11px;margin-top:6px;'>&#9888; Price looks like USD — yfinance fallback may be returning USD for this .NS ticker</div>"

    return f"""<div class='price-card {card_cls}'>
  <div style='display:flex;align-items:center;gap:8px;margin-bottom:10px;'>
    <span style='width:8px;height:8px;border-radius:50%;background:{dot_color};display:inline-block;'></span>
    <span style='color:#555;font-size:11px;letter-spacing:0.08em;'>{ms.upper()}</span>
    <span style='color:#333;font-size:11px;'>&#124;</span>
    <span style='color:#444;font-size:11px;'>{ts}</span>
    <span style='color:#333;font-size:11px;'>&#124;</span>
    <span style='color:#444;font-size:11px;'>{src}</span>
  </div>
  <div style='font-size:15px;font-weight:600;color:#aaa;letter-spacing:0.1em;margin-bottom:4px;'>{ticker}</div>
  <div style='font-size:38px;font-weight:700;color:white;line-height:1.1;margin-bottom:6px;'>
    {sym}{info['price']:,.2f}
  </div>
  <div style='font-size:16px;color:{clr};margin-bottom:10px;'>
    {arrow} {sign}{info['change']:.2f} &nbsp; <span style='font-size:14px;'>({sign}{info['change_pct']:.2f}%)</span>
  </div>
  <div style='display:flex;gap:20px;font-size:11px;color:#555;border-top:1px solid #1a1a2e;padding-top:8px;'>
    <span>Prev close: {sym}{info['prev_close']:,.2f}</span>
    <span>Vol: {vol}</span>
    <span>CCY: {info['currency']}</span>
  </div>
  <div style='font-size:10px;color:#333;margin-top:6px;'>
    Prices delayed ~15 min (yfinance free tier)
  </div>
  {currency_warning}
</div>"""


def _timer_tick() -> str:
    return refresh_live_ticker(_current_ticker["value"])


def _sync_state(t: str):
    _current_ticker["value"] = t.strip().upper() if t else CONFIG["ticker"]


# ─────────────────────────────────────────────────────────────────────────────
# PREDICTION
# ─────────────────────────────────────────────────────────────────────────────
def gradio_predict(ticker_input: str) -> str:
    ticker = (ticker_input or "").strip().upper() or CONFIG["ticker"]
    _current_ticker["value"] = ticker

    try:
        r   = predict_tomorrow(ticker)
        sym = r["symbol"]
        is_bull      = "BULL" in r["signal"]
        signal_color = "#4CAF50" if is_bull else "#F44336"
        pct_sign     = "+" if r["pct_change"] >= 0 else ""
        signal_label = "BULLISH" if is_bull else "BEARISH"

        model_rows = ""
        for label, key, color in [
            ("XGBoost",       "xgboost",       "#FF9800"),
            ("LightGBM",      "lightgbm",      "#64B5F6"),
            ("Random Forest", "random_forest",  "#81C784"),
        ]:
            diff     = r[key] - r["live_price"]
            diff_pct = diff / max(r["live_price"], 0.01) * 100
            d_sign   = "+" if diff >= 0 else ""
            d_color  = "#4CAF50" if diff >= 0 else "#F44336"
            model_rows += f"""
  <tr style='border-bottom:1px solid #1a1a2e;'>
    <td style='padding:7px 10px;color:#666;font-size:12px;'>{label}</td>
    <td style='padding:7px 10px;color:{color};font-weight:600;'>{sym}{r[key]:.2f}</td>
    <td style='padding:7px 10px;color:{d_color};font-size:11px;'>{d_sign}{diff_pct:.2f}% vs live</td>
  </tr>"""

        live_diff     = r["ensemble"] - r["live_price"]
        live_diff_pct = live_diff / max(r["live_price"], 0.01) * 100
        ld_sign       = "+" if live_diff >= 0 else ""
        ld_color      = "#4CAF50" if live_diff >= 0 else "#F44336"

        return f"""<div class='pred-card'>
  <div style='display:flex;align-items:baseline;gap:10px;margin-bottom:16px;flex-wrap:wrap;'>
    <span style='font-size:13px;font-weight:700;color:{signal_color};letter-spacing:0.1em;'>{signal_label}</span>
    <span style='font-size:13px;color:#555;'>&#8594;</span>
    <span style='font-size:22px;font-weight:700;color:white;'>{sym}{r['ensemble']:.2f}</span>
    <span style='font-size:14px;color:{signal_color};'>({pct_sign}{r['pct_change']:.2f}%)</span>
    <span style='font-size:11px;color:#444;margin-left:auto;'>{r['ticker']}</span>
  </div>

  <table style='width:100%;border-collapse:collapse;'>
    <tr style='border-bottom:1px solid #1a1a2e;'>
      <td style='padding:7px 10px;color:#555;font-size:12px;'>Live price</td>
      <td style='padding:7px 10px;color:white;font-weight:600;'>{sym}{r['live_price']:.2f}</td>
      <td style='padding:7px 10px;color:#F44336;font-size:11px;'>{r.get('change_pct_today',0):+.2f}% today</td>
    </tr>
    <tr style='border-bottom:1px solid #1a1a2e;'>
      <td style='padding:7px 10px;color:#555;font-size:12px;'>Last close</td>
      <td style='padding:7px 10px;color:#888;'>{sym}{r['current_price']:.2f}</td>
      <td style='padding:7px 10px;color:#444;font-size:11px;'>model base</td>
    </tr>
    {model_rows}
    <tr style='background:#0a0a1a;'>
      <td style='padding:9px 10px;color:#aaa;font-size:13px;font-weight:600;'>Ensemble</td>
      <td style='padding:9px 10px;color:{signal_color};font-size:16px;font-weight:700;'>{sym}{r['ensemble']:.2f}</td>
      <td style='padding:9px 10px;color:{ld_color};font-size:12px;'>{ld_sign}{live_diff_pct:.2f}% vs live</td>
    </tr>
  </table>

  <div style='margin-top:12px;font-size:10px;color:#333;border-top:1px solid #1a1a2e;padding-top:8px;'>
    Data: {r['last_data_date']} &nbsp;&#124;&nbsp; Market: {r['market_state']}
    &nbsp;&#124;&nbsp; Live injected: {r['live_injected']}
    &nbsp;&#124;&nbsp; For: {r['prediction_date']}
    &nbsp;&#124;&nbsp; Educational only
  </div>
</div>"""

    except Exception as e:
        import traceback
        return f"""<div class='pred-card' style='border-color:#3a1a1a;'>
  <div style='color:#F44336;'>Error: {e}</div>
  <pre style='color:#555;font-size:10px;margin-top:8px;white-space:pre-wrap;'>{traceback.format_exc()}</pre>
</div>"""


# ─────────────────────────────────────────────────────────────────────────────
# AGENT PIPELINE (wired into Gradio)
# ─────────────────────────────────────────────────────────────────────────────
def run_agents(ticker_input: str) -> tuple:
    """
    Runs the LangGraph agent pipeline and returns
    (final_decision_html, full_trace_html)
    """
    ticker = (ticker_input or "").strip().upper() or CONFIG["ticker"]

    # Check groq key
    if not KEYS.get('groq'):
        err = "<div class='agent-card' style='color:#FFB300;'>No GROQ_API_KEY set — add it to Colab Secrets to enable agent reasoning.</div>"
        return err, err

    try:
        ml_result = predict_tomorrow(ticker)

        # Run graph
        final = trading_graph.invoke({
            "ticker":           ticker,
            "ml_signal":        ml_result,
            "shap_features":    list(zip(feature_cols, shap_values[-1].tolist())),
            "news_sentiment":   float(sentiment_report.get('combined_sentiment', 0.0)),
            "technical_report": "",
            "sentiment_report": "",
            "bull_case":        "",
            "bear_case":        "",
            "final_decision":   "",
        })

        # Parse decision for color
        dec = final['final_decision'].upper()
        dec_color = "#4CAF50" if "BUY" in dec else ("#F44336" if "SELL" in dec else "#FFB300")

        decision_html = f"""<div class='agent-card'>
<div style='font-size:11px;color:#4a4a6a;letter-spacing:0.1em;margin-bottom:12px;'>PORTFOLIO MANAGER DECISION</div>
<div style='color:{dec_color};font-size:14px;font-weight:600;margin-bottom:12px;'>
{"BUY" if "BUY" in dec else ("SELL" if "SELL" in dec else "HOLD")}
</div>
<div style='color:#ccc;font-size:13px;line-height:1.8;'>{final['final_decision']}</div>
</div>"""

        trace_html = f"""<div class='agent-card'>
<div style='font-size:11px;color:#4a4a6a;letter-spacing:0.1em;margin-bottom:16px;'>FULL AGENT TRACE</div>

<div style='color:#64B5F6;font-size:11px;margin-bottom:4px;'>TECHNICAL ANALYST</div>
<div style='color:#aaa;margin-bottom:16px;'>{final['technical_report']}</div>

<div style='color:#FFB300;font-size:11px;margin-bottom:4px;'>SENTIMENT ANALYST</div>
<div style='color:#aaa;margin-bottom:16px;'>{final['sentiment_report']}</div>

<div style='color:#4CAF50;font-size:11px;margin-bottom:4px;'>BULL RESEARCHER</div>
<div style='color:#aaa;margin-bottom:16px;'>{final['bull_case']}</div>

<div style='color:#F44336;font-size:11px;margin-bottom:4px;'>BEAR RESEARCHER</div>
<div style='color:#aaa;margin-bottom:16px;'>{final['bear_case']}</div>

<div style='color:#CE93D8;font-size:11px;margin-bottom:4px;'>PORTFOLIO MANAGER</div>
<div style='color:#ccc;'>{final['final_decision']}</div>
</div>"""

        return decision_html, trace_html

    except Exception as e:
        import traceback
        err = f"<div class='agent-card' style='color:#F44336;'>Agent error: {e}<br><pre style='font-size:10px;'>{traceback.format_exc()}</pre></div>"
        return err, err


# ─────────────────────────────────────────────────────────────────────────────
# NSE WATCHLIST
# ─────────────────────────────────────────────────────────────────────────────
def refresh_indian_watchlist(_=None) -> str:
    tickers = [
        "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS",
        "WIPRO.NS", "ADANIENT.NS", "TATAMOTORS.NS", "BAJFINANCE.NS",
    ]
    rows = ""
    for t in tickers:
        info = get_live_price(t)
        if info.get("error"):
            rows += f"<tr><td colspan='5' style='padding:6px 8px;color:#F44336;font-size:11px;'>{t}: {info['error']}</td></tr>"
            continue

        sign  = "+" if info["change"] >= 0 else ""
        clr   = "#4CAF50" if info["change"] >= 0 else "#F44336"
        name  = t.replace(".NS","").replace(".BO","")
        price = info['price']
        sym   = "&#8377;"

        # Flag suspiciously low prices for Indian stocks (USD leak)
        price_flag = " &#9888;" if (price < 50 and info["currency"] == "INR") else ""

        rows += f"""<tr style='border-bottom:1px solid #0f0f1e;'>
  <td style='padding:6px 8px;color:#888;font-size:12px;'>{name}</td>
  <td style='padding:6px 8px;color:white;font-weight:600;font-size:12px;'>{sym}{price:,.2f}{price_flag}</td>
  <td style='padding:6px 8px;color:{clr};font-size:12px;'>{sign}{info['change']:.2f}</td>
  <td style='padding:6px 8px;color:{clr};font-size:11px;'>({sign}{info['change_pct']:.2f}%)</td>
  <td style='padding:6px 8px;color:#444;font-size:10px;'>{info.get("market_state","")}</td>
</tr>"""

    now = datetime.now().strftime("%H:%M:%S")
    return f"""<div class='watch-card'>
  <div style='font-size:11px;color:#4a4a6a;letter-spacing:0.1em;margin-bottom:12px;'>
    NSE LIVE &nbsp;&#124;&nbsp; auto-refresh 30s &nbsp;&#124;&nbsp; {now}
  </div>
  <table style='width:100%;border-collapse:collapse;'>
    <tr style='border-bottom:1px solid #1a1a2e;'>
      <th style='padding:5px 8px;color:#333;font-size:10px;text-align:left;font-weight:500;letter-spacing:0.08em;'>STOCK</th>
      <th style='padding:5px 8px;color:#333;font-size:10px;text-align:left;font-weight:500;letter-spacing:0.08em;'>PRICE</th>
      <th style='padding:5px 8px;color:#333;font-size:10px;text-align:left;font-weight:500;letter-spacing:0.08em;'>CHG</th>
      <th style='padding:5px 8px;color:#333;font-size:10px;text-align:left;font-weight:500;letter-spacing:0.08em;'>%</th>
      <th style='padding:5px 8px;color:#333;font-size:10px;text-align:left;font-weight:500;letter-spacing:0.08em;'>STATUS</th>
    </tr>
    {rows}
  </table>
  <div style='font-size:10px;color:#2a2a3a;margin-top:8px;'>
    Delayed ~15 min. NSE closes 15:30 IST. &#9888; flag = possible USD data leak.
  </div>
</div>"""


# ─────────────────────────────────────────────────────────────────────────────
# GRADIO LAYOUT
# ─────────────────────────────────────────────────────────────────────────────
LOADING_SPINNER = """<div style='background:#0d0d1a;border:1px solid #1e1e3a;border-radius:12px;
padding:24px;text-align:center;font-family:monospace;color:#444;'>
  Running pipeline...
</div>"""

with gr.Blocks(css=CUSTOM_CSS, title="Stock ML v4.1") as demo:

    gr.HTML("""
    <div style='padding:20px 0 8px;'>
      <div style='font-family:monospace;font-size:11px;color:#4a4a6a;
                  letter-spacing:0.15em;margin-bottom:6px;'>
        ML PIPELINE v4.1
      </div>
      <div style='font-size:22px;font-weight:700;color:white;margin-bottom:4px;'>
        Stock Price Prediction
      </div>
      <div style='font-size:12px;color:#444;font-family:monospace;'>
        XGBoost &middot; LightGBM &middot; Random Forest &middot; Ensemble &middot; LSTM
        &nbsp;&nbsp;|&nbsp;&nbsp;
        Yahoo Finance &middot; FinBERT &middot; FRED &middot; Finnhub
      </div>
    </div>
    """)

    # ── Input row ─────────────────────────────────────────────────────────────
    with gr.Row():
        tb = gr.Textbox(
            placeholder="AAPL  |  RELIANCE.NS  |  TCS.NS  |  MSFT  |  PLTR",
            value=CONFIG["ticker"],
            label="Ticker",
            scale=4,
        )
        predict_btn = gr.Button("Run Prediction", variant="primary", scale=1)
        agent_btn   = gr.Button("Run Agents",     variant="secondary", scale=1)

    # ── Two column layout ─────────────────────────────────────────────────────
    with gr.Row():

        # Left column: live price + prediction
        with gr.Column(scale=3):
            gr.HTML("<div class='section-label'>Live Price</div>")
            live_html = gr.HTML()

            gr.HTML("<div class='section-label'>Tomorrow's Forecast</div>")
            pred_html = gr.HTML()

        # Right column: NSE watchlist
        with gr.Column(scale=2):
            gr.HTML("<div class='section-label'>NSE Watchlist</div>")
            watch_html = gr.HTML()

    # ── Agent output (full width, below) ─────────────────────────────────────
    gr.HTML("<div class='section-label'>Agent Decision</div>")
    with gr.Row():
        agent_decision_html = gr.HTML()
        agent_trace_html    = gr.HTML()

    gr.HTML("""
    <div style='font-size:10px;color:#2a2a3a;font-family:monospace;
                border-top:1px solid #0f0f1e;margin-top:16px;padding-top:10px;'>
      Educational purposes only. Not financial advice.
      Prices delayed ~15 min (yfinance free tier).
    </div>
    """)

    # ── Wiring ────────────────────────────────────────────────────────────────

    # Predict button: spinner then result
    predict_btn.click(
        fn=lambda: LOADING_SPINNER, inputs=None, outputs=pred_html, queue=False
    ).then(
        fn=gradio_predict, inputs=tb, outputs=pred_html
    )

    # Agent button: spinner then agents
    agent_btn.click(
        fn=lambda: (LOADING_SPINNER, LOADING_SPINNER),
        inputs=None,
        outputs=[agent_decision_html, agent_trace_html],
        queue=False,
    ).then(
        fn=run_agents,
        inputs=tb,
        outputs=[agent_decision_html, agent_trace_html],
    )

    # Ticker change: instant live price update
    tb.change(fn=refresh_live_ticker, inputs=tb, outputs=live_html)
    tb.change(fn=_sync_state, inputs=tb, outputs=None, queue=False)

    # Timer: 30s auto-refresh
    try:
        timer = gr.Timer(30)
        timer.tick(fn=_timer_tick,             inputs=None, outputs=live_html)
        timer.tick(fn=refresh_indian_watchlist, inputs=None, outputs=watch_html)
        print("Timer enabled (30s)")
    except AttributeError:
        with gr.Row():
            gr.Button("Refresh Price").click(refresh_live_ticker, tb, live_html)
            gr.Button("Refresh Watchlist").click(refresh_indian_watchlist, None, watch_html)

    # Page load
    demo.load(fn=lambda: refresh_live_ticker(CONFIG["ticker"]), inputs=None, outputs=live_html)
    demo.load(fn=refresh_indian_watchlist, inputs=None, outputs=watch_html)

print("Launching Gradio v4.1...")
demo.launch(share=True, debug=False)

Timer enabled (30s)
Launching Gradio v4.1...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://049b095bc50d0054e1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 💾 Step 17 — Save All Artifacts

In [27]:
import zipfile, shutil

results_df=pd.DataFrame({'Actual':y_test,'Linear':y_pred_lr,'Ridge':y_pred_ridge,
    'XGBoost':y_pred_xgb,'LightGBM':y_pred_lgb,'Random_Forest':y_pred_rf,
    'Ensemble':y_pred_ensemble},index=test_dates)
results_df.to_csv('predictions.csv')
with open('tomorrow_prediction.json','w') as f: json.dump(tomorrow,f,indent=2)
with open('sentiment_report.json','w') as f: json.dump(sentiment_report,f,indent=2)
metrics_df.reset_index().to_json('metrics.json',orient='records',indent=2)
wf_results.to_csv('walkforward_results.csv',index=False)
with open('analysis_report.txt','w') as f: f.write(llm_report)

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    OUT='/content/drive/MyDrive/StockML_v3_Outputs/'
    os.makedirs(OUT,exist_ok=True)
    for fn in ['predictions.csv','metrics.json','tomorrow_prediction.json',
               'sentiment_report.json','walkforward_results.csv','analysis_report.txt',
               'shap_summary.png','shap_beeswarm.png','model_comparison.png',
               'xgboost_stock_model.json','lightgbm_stock_model.pkl',
               'random_forest_stock_model.pkl','lstm_stock_model.keras']:
        if os.path.exists(fn): shutil.copy(fn,OUT+fn); print(f'  ✅ {fn}')
    print(f'\n✅ All files saved to Google Drive: {OUT}')
except Exception as e:
    print(f'⚠️  Drive backup skipped: {e}')

# Zip for download
zn=f"{CONFIG['ticker']}_ML_v3_outputs.zip"
with zipfile.ZipFile(zn,'w',zipfile.ZIP_DEFLATED) as zf:
    for fn in ['predictions.csv','metrics.json','tomorrow_prediction.json',
               'shap_summary.png','model_comparison.png','analysis_report.txt']:
        if os.path.exists(fn): zf.write(fn)

print(f"\n{'='*65}")
print(f"  ✅ PIPELINE COMPLETE — {CONFIG['ticker']} v3.0")
print('='*65)
print(f"  Best RMSE    : {metrics_df['RMSE'].idxmin()} = {metrics_df['RMSE'].min():.6f}")
print(f"  Best DirAcc  : {metrics_df['Directional Acc %'].idxmax()} = {metrics_df['Directional Acc %'].max():.1f}%")
print(f"  Signal       : {tomorrow['signal']}")
print(f"  Ensemble     : {tomorrow['symbol']}{tomorrow['ensemble']:.2f} ({tomorrow['pct_change']:+.2f}%)")
print(f"  Live Price   : {tomorrow['symbol']}{tomorrow['live_price']:.2f} ({tomorrow['market_state']})")
print('='*65)
try:
    from google.colab import files; files.download(zn)
except: print(f'Download manually: {zn}')


  📡 RELIANCE.NS: source=yfinance_intraday price=INR 1311.6
  📡 AAPL: source=Finnhub price=USD 298.01
  📡 TCS.NS: source=yfinance_intraday price=INR 2136.3
  📡 INFY.NS: source=yfinance_intraday price=INR 1054.6
  📡 HDFCBANK.NS: source=yfinance_intraday price=INR 781.0
  📡 WIPRO.NS: source=yfinance_intraday price=INR 180.05
  📡 ADANIENT.NS: source=yfinance_intraday price=INR 3038.4


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['TATAMOTORS.NS']: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['TATAMOTORS.NS']: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")


  📡 BAJFINANCE.NS: source=yfinance_intraday price=INR 964.0
Mounted at /content/drive
  ✅ predictions.csv
  ✅ metrics.json
  ✅ tomorrow_prediction.json
  ✅ sentiment_report.json
  ✅ walkforward_results.csv
  ✅ analysis_report.txt
  ✅ shap_summary.png
  ✅ shap_beeswarm.png
  ✅ model_comparison.png
  ✅ xgboost_stock_model.json
  ✅ lightgbm_stock_model.pkl
  ✅ random_forest_stock_model.pkl
  ✅ lstm_stock_model.keras

✅ All files saved to Google Drive: /content/drive/MyDrive/StockML_v3_Outputs/

  ✅ PIPELINE COMPLETE — AAPL v3.0
  Best RMSE    : XGBoost = 0.017605
  Best DirAcc  : XGBoost = 51.2%
  Signal       : 📉 BEARISH
  Ensemble     : $296.83 (-0.40%)
  Live Price   : $298.01 (REGULAR)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>